# Proactive Churn - v1 feature selection

Reads the encoded model matrix `data/dataset_creation.ipynb` wrote into `data/raw/`, runs BorutaSHAP
on the `train` cell, and scores the full candidate set against the BorutaSHAP-accepted subset on the
two readable holdout cells. No Redshift connection - this notebook is offline.

**What it reads**, from `data/raw/` (gitignored), located by glob rather than by literal name because
the stem carries the extract's date range:

| glob | contents |
| --- | --- |
| `churn_v1_*_h*_encoded.parquet` | the model matrix - keys, label, `split`, passthroughs, raw categoricals, all of `model_cols` |
| `churn_v1_*_h*_encoding.json` | the encoding schema - `model_cols` in exact order, `categorical_features`, keys, passthroughs |

`model_cols` from the JSON **is** the feature list. This notebook does not re-derive it.

**What it produces**, in `data/raw/`:

| file | contents |
| --- | --- |
| `..._boruta_selection.json` | accepted / tentative / rejected lists + the run config (trials, sample size, seeds, source parquet) |
| `..._boruta_importance.csv` | the SHAP importance table - per feature mean / sd importance, decision, rank |
| `..._selection_eval.csv` | precision@k / recall@k / PR-AUC, full vs. selected, per holdout cell |
| `..._k_sweep.csv` | precision / recall / at-risk found per month / lift across `k` = 40 -> the whole month |
| `..._baseline_comparison.csv` | the model vs. the Retention Health Score at matched list sizes, plus the lead-list trigger rule |

**Split discipline.** Fit and select on `train` only. Read `holdout_school` and `holdout_time`.
`test` is the ship gate: it is touched once, later, and no cell here loads it. `embargo` and
`retrain_only` are excluded from everything. §6 asserts all of this rather than documenting it.

**Sections**

1. Imports
2. The `scipy.stats.binom_test` shim - runs BEFORE §3
3. Import BorutaShap
4. Locate the inputs
5. Load the matrix and the schema
6. Split discipline
7. `X` / `y` and the feature guardrails
8. Model configuration
9. BorutaSHAP
10. Persist the selection
11. Evaluation harness (11a. helpers, 11b. full vs. selected)
12. The `k` sweep and the operating point
13. Leakage / business-sense surface
14. Baseline comparison: model vs the Retention Health Score (14a. the rebuilt trend, 14b. three rankers, 14c. the trigger as used)
15. Audit, and what is deliberately not here

**Before you run anything:**

1. **Run `data/dataset_creation.ipynb` first** - §11 of it writes both input files. No VPN needed here.
2. **Kernel = "Python (Proactive Churn)"** - created by `setup.sh`. The pins in `requirements.txt`
   are load-bearing; §1 and §2 assert the two that this notebook can check.

Design decisions and the rationale behind them live in the Obsidian vault at
`Proactive Churn/Design Decisions.md`.

## 1. Imports

Modelling stack only - no `redshift_connector`, no `.env`. BorutaShap is **not** imported here; it
imports `scipy.stats.binom_test` at module level, so §2 has to run first.

The two pins this cell can check are asserted: `numpy<2` (BorutaShap uses `np.NaN`) and `xgboost<3`
(`shap`'s XGBoost tree loader cannot parse the 3.x `base_score` serialization). The third,
`scipy<1.14`, is §2's subject.

In [42]:
# --- stdlib ---
import ast          # §6 statically scans this notebook's own code cells
import json
import re
import time
from pathlib import Path

# --- data ---
import numpy as np
import pandas as pd

# --- modelling ---
import shap
import sklearn
import xgboost as xgb
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# --- the two pins that are checkable from here -----------------------------
# BorutaShap 1.0.17 (last released Dec 2023) is unmaintained. requirements.txt
# pins around three incompatibilities; these two cannot be shimmed at runtime, so
# fail here rather than several minutes into §9.
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} - BorutaShap 1.0.17 calls np.NaN in "
    "update_importance_history(), removed in numpy 2.0, so §9 dies partway "
    "through the trial loop. requirements.txt pins numpy>=1.26,<2. Fix the "
    "environment, then RESTART THE KERNEL."
)
assert int(xgb.__version__.split(".")[0]) < 3, (
    f"xgboost {xgb.__version__} - shap's XGBTreeModelLoader cannot parse the 3.x "
    "base_score serialization, and BorutaSHAP's importance measure is SHAP, so "
    "every trial fails. requirements.txt pins xgboost>=2.1,<3."
)

print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"xgboost      {xgb.__version__}")
print(f"shap         {shap.__version__}")
print("\nBorutaShap is imported in §3, AFTER §2's shim.")

Task was destroyed but it is pending!
task: <Task pending name='Task-379' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/swami/Desktop/Projects/Proactive_Churn/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-380' coro=<Kernel.shell_main() running at /Users/swami/Desktop/Projects/Proactive_Churn/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/swami/Desktop/Projects/Proactive_Churn/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:564]>
Task was destroyed but it is pending!
task: <Task pending name='Task-380' coro=<Kernel.shell_main() running at /Users/swami/Desktop/Projects/Proactive_Churn/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>
Task was destroyed but it is pending!
task: <Task pending name='Task-422' coro=<_async_in_context.<locals>.run_in_context() done, d

numpy        1.26.4
pandas       2.3.3
scikit-learn 1.9.0
xgboost      2.1.4
shap         0.49.1

BorutaShap is imported in §3, AFTER §2's shim.


## 2. The `scipy.stats.binom_test` shim

**Run this before §3.** BorutaShap 1.0.17 does `from scipy.stats import binom_test, ks_2samp` at
module level. `binom_test` was removed in scipy 1.12, and the project pins `scipy>=1.11,<1.14`, so
whether the name exists depends on which end of that range is installed. The shim defines it in terms
of `scipy.stats.binomtest` - returning the `.pvalue` - **only if it is absent**, and prints which of
the two states it found.

`BorutaShap.binomial_H0_test` passes the hit counts straight through, and those are floats from
`np.zeros`, so the shim casts `x` and `n` to `int` - `binomtest` rejects a float `k`. A known value
is asserted against the closed form before §3 imports anything.

This is documented in `README.md` alongside the pins.

In [43]:
import scipy
from scipy import stats as _st

BINOM_TEST_SHIM_NEEDED = not hasattr(_st, "binom_test")

if BINOM_TEST_SHIM_NEEDED:

    def binom_test(x, n=None, p=0.5, alternative="two-sided"):
        """scipy.stats.binom_test (removed in scipy 1.12) via binomtest.

        Signature-compatible with what BorutaShap 1.0.17 calls:
        binom_test(x, n=n, p=0.5, alternative='greater' | 'less').

        x and n are cast to int because BorutaShap passes the hit counters, which
        are floats out of np.zeros / np.vstack, and binomtest rejects a float k.
        """
        return float(_st.binomtest(int(x), n=int(n), p=p, alternative=alternative).pvalue)

    _st.binom_test = binom_test

# Whichever branch ran, the name has to be importable from scipy.stats now -
# BorutaShap's module-level import is what consumes it.
assert hasattr(scipy.stats, "binom_test"), (
    "scipy.stats.binom_test is still missing, so §3's `from BorutaShap import "
    "BorutaShap` will raise ImportError. The shim assigns onto the module object "
    "`scipy.stats`; if that is not the module BorutaShap imports from, something "
    "has shadowed it in this kernel - restart and re-run §1 -> §2 -> §3 in order."
)

# 8 or more heads in 10 fair tosses: sum(C(10,k)/2^10 for k in 8..10) = 56/1024.
_p = scipy.stats.binom_test(8, n=10, p=0.5, alternative="greater")
assert np.isclose(_p, 56 / 1024), (
    f"binom_test(8, n=10, p=0.5, alternative='greater') returned {_p!r}, expected "
    f"{56 / 1024}. A wrong one-sided p-value would not raise in §9 - it would "
    "silently change which features BorutaSHAP accepts."
)

print(f"scipy {scipy.__version__}")
print(f"shim  {'INSTALLED - scipy.stats had no binom_test' if BINOM_TEST_SHIM_NEEDED else 'NOT NEEDED - scipy.stats.binom_test exists natively'}")
print(f"check binom_test(8, n=10, p=.5, 'greater') = {_p:.10f} == 56/1024  OK")

scipy 1.13.1
shim  NOT NEEDED - scipy.stats.binom_test exists natively
check binom_test(8, n=10, p=.5, 'greater') = 0.0546875000 == 56/1024  OK


## 3. Import BorutaShap

Imports the package and pins its API by assertion, because the signature is unusual in two ways that
matter and neither raises anything readable if it drifts:

- `importance_measure` must be the **lowercase** `"shap"`. The constructor's own default is
  `"Shap"`, and `feature_importance()` compares `== "shap"`, so the default value raises
  `ValueError: No Importance_measure was specified` on the first trial.
- `fit()` takes `n_trials`, `random_state`, `sample`, `train_or_test`, `normalize`, `verbose` and
  `stratify` - not the scikit-learn-shaped signature the class's `get_params` / `set_params` imply.

`train_or_test` is BorutaShap's own in-sample / out-of-sample switch for the SHAP computation and has
**nothing to do with this project's `test` split cell**; §9 passes `"train"`.

In [44]:
import inspect

from BorutaShap import BorutaShap

# LOWERCASE. The constructor default is "Shap"; feature_importance() compares
# `self.importance_measure == "shap"` and raises ValueError on anything else, so
# the default breaks the class.
IMPORTANCE_MEASURE = "shap"
assert IMPORTANCE_MEASURE == "shap", (
    "IMPORTANCE_MEASURE must be the lowercase 'shap'. BorutaShap's constructor "
    "defaults to 'Shap' and its feature_importance() does an exact "
    "`== 'shap'` comparison, so the capitalised default raises "
    "ValueError('No Importance_measure was specified select one of (shap, perm, "
    "gini)') on the first trial - after the first model fit, several minutes in."
)

_init_params = set(inspect.signature(BorutaShap.__init__).parameters)
_fit_params = set(inspect.signature(BorutaShap.fit).parameters)

_expected_init = {"model", "importance_measure", "classification", "percentile", "pvalue"}
_expected_fit = {"X", "y", "sample_weight", "n_trials", "random_state", "sample",
                 "train_or_test", "normalize", "verbose", "stratify"}

assert _expected_init <= _init_params, (
    f"BorutaShap.__init__ is missing {sorted(_expected_init - _init_params)}. The "
    "installed version is not the pinned 1.0.17 and §9 passes keywords it does "
    "not accept."
)
assert _expected_fit <= _fit_params, (
    f"BorutaShap.fit is missing {sorted(_expected_fit - _fit_params)}. The "
    "installed version is not the pinned 1.0.17."
)

print(f"BorutaShap.__init__{inspect.signature(BorutaShap.__init__)}")
print(f"BorutaShap.fit{inspect.signature(BorutaShap.fit)}")
print(f"\nimportance_measure = {IMPORTANCE_MEASURE!r} (lowercase - the 'Shap' default raises)")

BorutaShap.__init__(self, model=None, importance_measure='Shap', classification=True, percentile=100, pvalue=0.05)
BorutaShap.fit(self, X, y, sample_weight=None, n_trials=20, random_state=0, sample=False, train_or_test='test', normalize=True, verbose=True, stratify=None)

importance_measure = 'shap' (lowercase - the 'Shap' default raises)


## 4. Locate the inputs

Globs `data/raw/` for the encoded parquet and the encoding JSON and demands **exactly one** of each.
The stem embeds the extract's observation window (`churn_v1_is_reportable_request_202401_202605_h3`),
so it changes whenever `AS_OF_DATE` or `OBS_START` moves and must not be hard-coded here.

Zero matches and more than one match both fail with a message naming the directory and what it found:
zero means the extract has not been run, and two means an old extract is still sitting there and
picking one silently would make every number below untraceable.

`STEM` is derived from the parquet's own name and is what §10 onwards name their outputs after, so
every artefact in `data/raw/` sorts next to the extract it came from.

In [45]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "data" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# This notebook's own file - §6 reads it to scan its code cells.
NB_PATH = DATA_DIR / "feature_selection.ipynb"

# Globs, not names: the stem carries the date range the extract was built over.
ENCODED_GLOB = "churn_v1_*_h*_encoded.parquet"
ENCODING_GLOB = "churn_v1_*_h*_encoding.json"


def one_match(directory: Path, pattern: str) -> Path:
    """The single file in `directory` matching `pattern`, or a clear failure.

    Zero matches and 2+ matches are both fatal: zero means the extract was never
    run, and picking one of several silently would leave the run untraceable.
    """
    if not directory.is_dir():
        raise FileNotFoundError(
            f"no such directory: {directory}. It is created by "
            "data/dataset_creation.ipynb §2 and is gitignored, so a fresh clone "
            "has neither the directory nor the extract. Run the extract notebook "
            "first."
        )

    hits = sorted(directory.glob(pattern))

    if not hits:
        _present = sorted(p.name for p in directory.iterdir() if p.is_file())
        raise FileNotFoundError(
            f"no file matching {pattern!r} in {directory}. Run "
            "data/dataset_creation.ipynb through §11 - it writes the encoded "
            f"parquet and the encoding JSON. Files present ({len(_present)}): "
            f"{_present[:12]}{' ...' if len(_present) > 12 else ''}"
        )

    if len(hits) > 1:
        raise ValueError(
            f"{len(hits)} files match {pattern!r} in {directory}: "
            f"{[p.name for p in hits]}. The stem carries the extract's date "
            "range, so more than one match means an older extract is still "
            "there. Choosing one here would make every number in this notebook "
            "untraceable - move or delete the stale extract (both the parquet "
            "and its JSON), then re-run this cell."
        )

    return hits[0]


ENCODED = one_match(RAW_DIR, ENCODED_GLOB)
ENCODING = one_match(RAW_DIR, ENCODING_GLOB)

_SUFFIX = "_encoded.parquet"
assert ENCODED.name.endswith(_SUFFIX), f"unexpected parquet name: {ENCODED.name}"
STEM = ENCODED.name[: -len(_SUFFIX)]

assert ENCODING.name == f"{STEM}_encoding.json", (
    f"the two inputs do not come from the same extract: parquet {ENCODED.name!r} "
    f"has stem {STEM!r}, but the JSON is {ENCODING.name!r} (expected "
    f"{STEM}_encoding.json). §11 of the extract derives both names from one stem, "
    "so a mismatch means the two files are from different runs and model_cols may "
    "not describe this matrix."
)

print(f"raw dir   {RAW_DIR}")
print(f"parquet   {ENCODED.name}  ({ENCODED.stat().st_size / 1e6:.1f} MB)")
print(f"encoding  {ENCODING.name}  ({ENCODING.stat().st_size / 1e3:.1f} KB)")
print(f"stem      {STEM}")

raw dir   /Users/swami/Desktop/Projects/Proactive_Churn/data/raw
parquet   churn_v1_is_reportable_request_202401_202605_h3_encoded.parquet  (114.5 MB)
encoding  churn_v1_is_reportable_request_202401_202605_h3_encoding.json  (9.5 KB)
stem      churn_v1_is_reportable_request_202401_202605_h3


## 5. Load the matrix and the schema

Reads the JSON first and the parquet second, and names `MODEL_COLS` from the sidecar. **`model_cols`
is the feature list**; nothing here re-derives it from the parquet's columns, because a list rebuilt
by pattern would silently disagree with the one a scoring job replays.

The two artefacts are then made to agree, in this order: every `model_cols` entry exists in the
parquet (a name in the JSON that the matrix does not have is the failure that would otherwise surface
as a `KeyError` in §7), no duplicates, and `model_cols` is the tail of the column list in order.

Prints the row count, the schema's categorical features with their frozen train levels, and the
`split` census.

In [46]:
SCHEMA = json.loads(ENCODING.read_text())

MODEL_COLS_FILE = list(SCHEMA["model_cols"])       # exactly what the file carries
MODEL_COLS = list(MODEL_COLS_FILE)                 # authoritative, not re-derived

# --- late drops -------------------------------------------------------------
# Escape hatch for a feature found to be unusable AFTER the extract ran, so a
# selection re-run costs seconds instead of a re-extract. The permanent home for
# an exclusion is EXCLUDE_PATTERNS in dataset_creation.ipynb; anything listed
# here should already be there, and once the extract is re-run this list becomes
# a no-op (it drops nothing and says so).
#
# fee_percentage: separated cancellers at AUC ~0.85 within a 2024 month and
# ~0.50 within a 2025 month. Expired, and it dominated the model while it lasted.
DROP_FEATURE_PATTERNS = [r"fee_percentage"]

_dropped_late = [c for c in MODEL_COLS
                 if any(re.search(p, c) for p in DROP_FEATURE_PATTERNS)]
if _dropped_late:
    MODEL_COLS = [c for c in MODEL_COLS if c not in set(_dropped_late)]
    print(f"DROP_FEATURE_PATTERNS removed {len(_dropped_late)} of "
          f"{len(_dropped_late) + len(MODEL_COLS)} feature(s) before selection:")
    for c in _dropped_late:
        print(f"    {c}")
    print("  -> the extract still carries them; they are excluded from X only.")
else:
    print("DROP_FEATURE_PATTERNS matched nothing - the extract already excludes them.")

CATEGORICAL_FEATURES = dict(SCHEMA["categorical_features"])
KEYS = list(SCHEMA["keys"])
Y_COL = SCHEMA["y_col"]
PASSTHROUGH = list(SCHEMA["passthrough"])
NON_FEATURE_CARRIED = list(SCHEMA.get("non_feature_carried", []))
DUMMY_SEP = SCHEMA.get("dummy_separator", "__")

enc = pd.read_parquet(ENCODED)
enc["dim_month"] = pd.to_datetime(enc["dim_month"])

# --- do the JSON and the parquet describe the same matrix? ------------------
_missing = [c for c in MODEL_COLS if c not in enc.columns]
assert not _missing, (
    f"{len(_missing)} column(s) in the JSON's model_cols are NOT in "
    f"{ENCODED.name}: {_missing[:20]}{' ...' if len(_missing) > 20 else ''}\n"
    "  The sidecar and the parquet are from different runs, or the parquet was "
    "rewritten by hand. model_cols is the contract a scoring job replays, so "
    "this notebook refuses to select features against a matrix that does not "
    "match it. Re-run data/dataset_creation.ipynb §11 to regenerate both."
)

_dupes = sorted({c for c in MODEL_COLS if MODEL_COLS.count(c) > 1})
assert not _dupes, (
    f"duplicate entries in the JSON's model_cols: {_dupes}. A booster fitted on "
    "this list and a matrix selected by it would disagree about width."
)

# Checked against the UNFILTERED list from the sidecar: the layout contract is a
# property of the file, not of whatever this notebook then chooses to model on.
assert list(enc.columns)[-len(MODEL_COLS_FILE):] == MODEL_COLS_FILE, (
    "model_cols is not the tail of the parquet's column list, in order. §11 of "
    "the extract asserts that it is, so the parquet has been rewritten since. "
    "Column order is part of the encoding contract - regenerate rather than "
    "reordering here."
)

_required = set(KEYS) | {Y_COL, "split", "school_group"}
_absent = sorted(_required - set(enc.columns))
assert not _absent, (
    f"the encoded parquet is missing {_absent}, which this notebook needs for "
    "cell selection and per-month metrics. Regenerate it from the extract."
)

print(f"{len(enc):,} rows x {enc.shape[1]} columns")
print(f"MODEL_COLS            {len(MODEL_COLS)}")
print(f"KEYS                  {KEYS}")
print(f"Y_COL                 {Y_COL}")
print(f"PASSTHROUGH           {PASSTHROUGH}   (never features)")
print(f"NON_FEATURE_CARRIED   {NON_FEATURE_CARRIED}")
print(f"months                {enc['dim_month'].min():%Y-%m} -> {enc['dim_month'].max():%Y-%m}")
print(f"schools               {enc['school_id'].nunique():,}")
print(f"positives             {int(enc[Y_COL].sum()):,} ({enc[Y_COL].mean():.2%} of rows)")

print(f"\ncategorical features encoded by the extract ({len(CATEGORICAL_FEATURES)}), "
      "levels frozen from `train`:")
for _c, _levels in CATEGORICAL_FEATURES.items():
    print(f"  {_c:<36}{len(_levels):>3} level(s) -> {len(_levels) + 2} dummies "
          f"(+{DUMMY_SEP}other, +{DUMMY_SEP}isna)")

_dummy_cols = [c for c in MODEL_COLS
               if any(c.startswith(f"{k}{DUMMY_SEP}") for k in CATEGORICAL_FEATURES)]
print(f"\nMODEL_COLS = {len(MODEL_COLS) - len(_dummy_cols)} numeric + {len(_dummy_cols)} dummies")

DROP_FEATURE_PATTERNS matched nothing - the extract already excludes them.
800,161 rows x 186 columns
MODEL_COLS            177
KEYS                  ['school_id', 'dim_month']
Y_COL                 label
PASSTHROUGH           ['product_score']   (never features)
NON_FEATURE_CARRIED   ['split', 'school_group']
months                2024-01 -> 2026-05
schools               44,043
positives             24,300 (3.04% of rows)

categorical features encoded by the extract (3), levels frozen from `train`:
  customer_segment                      4 level(s) -> 6 dummies (+__other, +__isna)
  school_country                        2 level(s) -> 4 dummies (+__other, +__isna)
  monthly_billing_status_local          6 level(s) -> 8 dummies (+__other, +__isna)

MODEL_COLS = 159 numeric + 18 dummies


---

## 6. Split discipline

The 2x2 cells the extract wrote onto `split`. This notebook's rules, and how each one is enforced
rather than described:

| cell | role here | enforced by |
| --- | --- | --- |
| `train` | the only cell fitted on, and the only cell BorutaSHAP sees | `cell()` is the only row selector |
| `holdout_school` | read: new schools, same era. Half its schools carve §8's early-stopping eval set | `cell()`; the two halves are asserted school-disjoint in §8 |
| `holdout_time` | read: same schools, new era | `cell()` |
| `test` | **the ship gate.** Touched once, later, by a different notebook. No cell here loads it | `cell()` refuses it; §6's static scan; §15's access log |
| `embargo` | excluded | `cell()` refuses it |
| `retrain_only` | excluded | `cell()` refuses it |

Three enforcement mechanisms, because "documented" is not "enforced":

1. **`cell(split)` is the only way this notebook selects rows** and it raises on anything outside
   `LOADABLE_SPLITS`. Every access is appended to `SPLIT_ACCESS_LOG`.
2. **A static scan of this notebook's own code cells** (`ast`, over `NB_PATH`) fails if any cell
   compares against the literal `"test"` outside an `assert`, or passes it to a call. A comparison
   inside an assertion is the *point* - a filter is not.
3. **§15 re-reads `SPLIT_ACCESS_LOG`** at the end of the run and asserts the ship gate is absent.

This cell also **re-verifies school-disjointness from the file** - `train` against `test` and
`train` against `holdout_school`. The extract asserts it, but an input that has been regenerated,
edited or partially overwritten should not be trusted silently. The check reads `school_id` and
`split` only; no feature of a `test` row is ever loaded.

In [47]:
FIT_SPLIT = "train"
READ_SPLITS = ("holdout_school", "holdout_time")
SHIP_GATE_SPLIT = "test"                       # named so it can be asserted against
EXCLUDED_SPLITS = ("embargo", "retrain_only")

LOADABLE_SPLITS = (FIT_SPLIT,) + READ_SPLITS
NEVER_LOADED = (SHIP_GATE_SPLIT,) + EXCLUDED_SPLITS
ALL_SPLITS = LOADABLE_SPLITS + NEVER_LOADED

_WHY_NOT_LOADABLE = {
    SHIP_GATE_SPLIT: (
        "the ship gate - new schools in a new era, the honest deployment number. "
        "It is read ONCE, later, on a model that is already final. Every look at "
        "it spends a little of its value; a look during feature selection spends "
        "all of it, because the selection is then fitted to it."
    ),
    "embargo": (
        "the months between the training window and the evaluation window, "
        "excluded so no training row's label window straddles the seam. Labelled "
        "rather than deleted by the extract so the cost is visible."
    ),
    "retrain_only": (
        "answerable months after the evaluation window, kept for the live "
        "retrain. Never evaluated."
    ),
}

SPLIT_ACCESS_LOG = []


def cell(split: str, cols=None) -> pd.DataFrame:
    """Rows of one split cell. The ONLY row selector in this notebook.

    Raises on any cell that must not be loaded, and logs every access into
    SPLIT_ACCESS_LOG so §15 can audit what this run actually touched.
    """
    if split not in LOADABLE_SPLITS:
        _why = _WHY_NOT_LOADABLE.get(split, "not a split cell of this dataset")
        raise ValueError(
            f"cell({split!r}) refused. Loadable here: {list(LOADABLE_SPLITS)}.\n"
            f"  {split!r}: {_why}\n"
            "  If you need this cell, you are writing a different notebook."
        )

    SPLIT_ACCESS_LOG.append(split)
    frame = enc.loc[enc["split"] == split]
    return frame if cols is None else frame[cols]


def Xy(split: str):
    """(X, y) for one cell: X is exactly MODEL_COLS, in order; y is int8."""
    frame = cell(split)
    return frame[MODEL_COLS], frame[Y_COL].astype("int8")


# --- the census, over every cell in the file --------------------------------
# Counted from `split` / `school_id` / the label only. Reading how many rows the
# ship gate has is not the same as loading its features, and the number is needed
# to state what is being left on the table.
_census = (enc.assign(_s=pd.Categorical(enc["split"], categories=ALL_SPLITS, ordered=True))
              .groupby("_s", observed=False)
              .agg(rows=(Y_COL, "size"),
                   schools=("school_id", "nunique"),
                   positives=(Y_COL, "sum"),
                   pos_rate=(Y_COL, "mean"),
                   first_month=("dim_month", "min"),
                   last_month=("dim_month", "max")))
_census["pos_rate"] = (_census["pos_rate"] * 100).round(2).astype(str) + "%"
_census["first_month"] = _census["first_month"].dt.strftime("%Y-%m")
_census["last_month"] = _census["last_month"].dt.strftime("%Y-%m")
_census["role"] = ["FIT + SELECT" if s == FIT_SPLIT
                   else "read" if s in READ_SPLITS
                   else "SHIP GATE - not loaded" if s == SHIP_GATE_SPLIT
                   else "excluded"
                   for s in ALL_SPLITS]
_census.index.name = "split"
print(_census.to_string())

_unexpected = sorted(set(enc["split"].astype(str).unique()) - set(ALL_SPLITS))
assert not _unexpected, (
    f"the parquet carries split value(s) this notebook does not know: "
    f"{_unexpected}. The extract writes exactly {list(ALL_SPLITS)}; an extra "
    "value means the split cell changed and the roles above may be wrong."
)
for _s in LOADABLE_SPLITS:
    assert (enc["split"] == _s).any(), (
        f"the {_s!r} cell is EMPTY in {ENCODED.name}. Nothing below can run - "
        "check §9 of the extract, which prints the per-cell composition."
    )

# --- re-verify school-disjointness FROM THE FILE ---------------------------
# Identifiers only: no feature of any non-loadable row is read here.
_ids_in = enc.groupby("split", observed=True)["school_id"].agg(set).to_dict()
for _a, _b in ((FIT_SPLIT, SHIP_GATE_SPLIT), (FIT_SPLIT, "holdout_school")):
    _shared = _ids_in.get(_a, set()) & _ids_in.get(_b, set())
    assert not _shared, (
        f"{len(_shared):,} school_id(s) appear in BOTH the {_a!r} and {_b!r} "
        f"cells, e.g. {sorted(_shared)[:5]}. The extract's §9 asserts the A/B "
        "school partition is disjoint, so this file has been regenerated with a "
        "different rule, edited, or concatenated from two runs. Feature "
        "selection on it would leak the held-out schools."
    )
print(f"\nschool-disjoint, re-verified from the file: "
      f"{FIT_SPLIT} n {SHIP_GATE_SPLIT} = 0, {FIT_SPLIT} n holdout_school = 0  OK")

# --- the static scan of this notebook's own code cells ---------------------


def scan_for_forbidden_split_use(nb_path: Path, forbidden):
    """Code cells that use a forbidden split value outside an assertion.

    Returns a list of (cell index, source line) violations. A forbidden value
    is allowed to appear in an `assert` (an assertion ABOUT the ship gate is
    the point), in a comment (invisible to ast) and in markdown. It is a
    violation anywhere else that could act as a filter: a comparison, or an
    argument to a call.
    """
    nb = json.loads(nb_path.read_text())
    violations = []

    for i, c in enumerate(nb["cells"]):
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        tree = ast.parse(src)

        allowed = set()
        for node in ast.walk(tree):
            if isinstance(node, ast.Assert):
                allowed.update(id(n) for n in ast.walk(node))

        for node in ast.walk(tree):
            if id(node) in allowed:
                continue
            if isinstance(node, ast.Compare):
                _operands = [node.left, *node.comparators]
                if any(isinstance(o, ast.Constant) and o.value in forbidden
                       for o in _operands):
                    violations.append((i, ast.unparse(node)))
            elif isinstance(node, ast.Call):
                _args = list(node.args) + [k.value for k in node.keywords]
                if any(isinstance(a, ast.Constant) and a.value in forbidden
                       for a in _args):
                    violations.append((i, ast.unparse(node)[:120]))

    return violations


if NB_PATH.exists():
    _violations = scan_for_forbidden_split_use(NB_PATH, set(NEVER_LOADED))
    assert not _violations, (
        f"{len(_violations)} code-cell use(s) of a non-loadable split value "
        f"{list(NEVER_LOADED)} outside an assertion - i.e. something that can "
        "act as a filter or be handed to a fit:\n"
        + "\n".join(f"    cell {i}: {s}" for i, s in _violations)
        + f"\n  The ship gate {SHIP_GATE_SPLIT!r} may be named in an assertion, a "
          "comment or markdown, never in a comparison or a call. Route every row "
          "selection through cell()."
    )
    print(f"static scan of {NB_PATH.name}: no code cell compares against, or passes, "
          f"any of {list(NEVER_LOADED)} outside an assertion  OK")
else:
    raise FileNotFoundError(
        f"cannot find this notebook at {NB_PATH} to scan it. NB_PATH is derived "
        "in §4 from PROJECT_ROOT, which assumes the notebook lives in "
        "<project>/data/ and is run from there or from the project root. Point "
        "NB_PATH at this file and re-run - the scan is one of the three things "
        "keeping the ship gate closed, so it is not optional."
    )

                  rows  schools  positives pos_rate first_month last_month                    role
split                                                                                             
train           209694    21893       6829    3.26%     2024-01    2024-12            FIT + SELECT
holdout_school   90207     9435       2966    3.29%     2024-01    2024-12                    read
holdout_time    184037    24109       5287    2.87%     2025-04    2025-12                    read
test             78474    10317       2349    2.99%     2025-04    2025-12  SHIP GATE - not loaded
embargo          81709    28495       2280    2.79%     2025-01    2025-03                excluded
retrain_only    156040    33949       4589    2.94%     2026-01    2026-05                excluded

school-disjoint, re-verified from the file: train n test = 0, train n holdout_school = 0  OK
static scan of feature_selection.ipynb: no code cell compares against, or passes, any of ['test', 'embargo', 'retr

---

## 7. `X` / `y` and the feature guardrails

Builds the `train` matrix through `cell()` and asserts what it is:

- `X` is **exactly** `MODEL_COLS`, same names, same order. Not a superset, not a reordering.
- **None** of `school_id`, `dim_month`, `split`, `school_group`, `label`, `product_score`, the
  schema's passthroughs, or anything that looks like an organization identifier is in it. The last is
  a pattern match over the parquet's columns, so an org column added later cannot arrive unnoticed.
- Every column is numeric. A surviving `category` dtype here would mean the extract's encoding did
  not cover a categorical feature, and XGBoost would either refuse it or read integer codes.

NaNs are **passed to XGBoost as-is** - it learns a missing branch natively, and for the `delta_*`
ratios a null ("the baseline was zero") and a `-1.0` ("the metric fell to zero") are opposite facts
that any fill would merge. No imputation anywhere in this notebook.

In [48]:
X_train, y_train = Xy(FIT_SPLIT)

assert list(X_train.columns) == MODEL_COLS, (
    "X is not exactly MODEL_COLS in order. Xy() selects frame[MODEL_COLS], so "
    "this can only fail if MODEL_COLS was mutated after §5."
)

# --- what may never be a feature -------------------------------------------
# Pattern as well as name, so an organization column added to the extract later
# cannot slip in silently.
ORG_PATTERN = re.compile(r"organi[sz]ation|(?:^|_)orgs?(?:_|$)")
ORG_LIKE = sorted(c for c in enc.columns if ORG_PATTERN.search(c))

FORBIDDEN_IN_X = (
    set(KEYS) | {Y_COL, "split", "school_group", "product_score"}
    | set(PASSTHROUGH) | set(NON_FEATURE_CARRIED) | set(CATEGORICAL_FEATURES)
    | set(ORG_LIKE)
)

_WHY_FORBIDDEN = {
    "school_id": "a key the trees would memorise",
    "dim_month": "the observation month - a clock, and the split boundary is a date",
    "split": "the train/test assignment itself; encoding it makes every off-train metric look wonderful",
    "school_group": "the other half of the split assignment",
    Y_COL: "the target",
    "product_score": "the Retention Health Score this model is scored AGAINST - circular",
}

_bad = sorted(set(MODEL_COLS) & FORBIDDEN_IN_X)
assert not _bad, (
    f"column(s) that must never be a feature are in MODEL_COLS: {_bad}\n"
    + "\n".join(
        f"    {c}: {_WHY_FORBIDDEN.get(c, 'a passthrough / identifier - carried in the frame, never an input')}"
        for c in _bad
    )
    + "\n  MODEL_COLS comes from the encoding JSON, so it was widened upstream: "
      "check §7b / §11.4 of data/dataset_creation.ipynb. Do not filter it here - "
      "the sidecar is the contract a scoring job replays, and patching it in this "
      "notebook would leave the two disagreeing."
)

_non_numeric = [(c, str(X_train[c].dtype)) for c in MODEL_COLS
                if not pd.api.types.is_numeric_dtype(X_train[c])]
assert not _non_numeric, (
    "MODEL_COLS column(s) are not numeric:\n"
    + "\n".join(f"    {c:<52} {d}" for c, d in _non_numeric)
    + "\n  A `category` here means the extract's one-hot encoding missed a "
      "categorical feature; XGBoost either refuses it or silently reads integer "
      "codes as measurements."
)

_pos = int(y_train.sum())
_neg = len(y_train) - _pos
_nan_cells = int(X_train.isna().to_numpy().sum())
_nan_cols = int((X_train.isna().sum() > 0).sum())

print(f"X_train   {X_train.shape[0]:,} rows x {X_train.shape[1]} columns "
      f"({cell(FIT_SPLIT)['school_id'].nunique():,} schools)")
print(f"y_train   {_pos:,} positive / {_neg:,} negative ({y_train.mean():.2%})")
print(f"dtypes    {sorted(set(X_train.dtypes.astype(str)))}")
print(f"NaN       {_nan_cells:,} cells across {_nan_cols} of {X_train.shape[1]} columns "
      f"({_nan_cells / X_train.size:.2%}) - passed to XGBoost as-is, never imputed")
print(f"\nforbidden-in-X set ({len(FORBIDDEN_IN_X)} names) held out of MODEL_COLS  OK")
print(f"  organization-like columns in the file: {ORG_LIKE or 'none'}")

X_train   209,694 rows x 177 columns (21,893 schools)
y_train   6,829 positive / 202,865 negative (3.26%)
dtypes    ['float32', 'uint8']
NaN       11,835,351 cells across 101 of 177 columns (31.89%) - passed to XGBoost as-is, never imputed

forbidden-in-X set (9 names) held out of MODEL_COLS  OK
  organization-like columns in the file: none


---

## 8. Model configuration

Two configurations off one set of base parameters, and the difference is load-bearing.

**`boruta_model()` - fixed `n_estimators`, NO early stopping.** BorutaShap's `Train_model` calls
`self.model.fit(X, y, sample_weight=..., verbose=False)` and supplies **no** `eval_set`, once per
trial, on a re-shadowed matrix. An estimator carrying `early_stopping_rounds` therefore raises
(`Must have at least 1 validation dataset for early stopping`) or, worse, silently stops on
whatever it can find. The cell asserts `early_stopping_rounds is None` on the estimator it builds.

**`eval_model()` - the same base parameters plus `early_stopping_rounds` on AUCPR**, with an explicit
eval set: half the schools of `holdout_school`, carved here, seeded, and asserted disjoint from the
half §11 reports on. The ship gate is not involved in either.

Shallow trees (`max_depth=3`) with a `min_child_weight` floor, `tree_method="hist"`,
`eval_metric="aucpr"`, a fixed `random_state` and `n_jobs=-1`.

`SCALE_POS_WEIGHT` is the **`train` cell's own** `neg/pos` ratio. The spec says to tune it *downward*
on AUCPR - over-weighting positives hurts precision and inflates the probabilities - and that tuning
is not in this notebook. Weighting applies to **training only**: no evaluation cell is reweighted, or
precision@k stops meaning anything.

In [49]:
RANDOM_STATE = 42

# --- the early-stopping eval set: half of holdout_school's SCHOOLS ----------
# School-level, so no school contributes to both the early-stopping set and the
# reported one. Drawn from a holdout cell, never from the ship gate.
ES_SPLIT = "holdout_school"
ES_SCHOOL_FRACTION = 0.50
ES_SEED = 7

_hs = cell(ES_SPLIT)
_hs_schools = np.array(sorted(_hs["school_id"].unique()))
_perm = np.random.default_rng(ES_SEED).permutation(len(_hs_schools))
_n_es = int(round(ES_SCHOOL_FRACTION * len(_hs_schools)))
ES_SCHOOLS = set(_hs_schools[_perm[:_n_es]])
READ_SCHOOLS_HS = set(_hs_schools[_perm[_n_es:]])

assert not (ES_SCHOOLS & READ_SCHOOLS_HS), "the two holdout_school halves share a school"
assert ES_SCHOOLS and READ_SCHOOLS_HS, (
    f"{ES_SPLIT} has {len(_hs_schools)} school(s) - too few to carve an "
    "early-stopping half and still have something to report on. Either widen the "
    "cell or drop early stopping and fix n_estimators."
)

ES_FRAME = _hs[_hs["school_id"].isin(ES_SCHOOLS)]
HS_READ_FRAME = _hs[_hs["school_id"].isin(READ_SCHOOLS_HS)]

X_es, y_es = ES_FRAME[MODEL_COLS], ES_FRAME[Y_COL].astype("int8")
assert int(y_es.sum()) > 0, (
    "the early-stopping eval set contains no positive rows, so AUCPR on it is "
    "undefined and early stopping would pick iteration 0. Raise "
    "ES_SCHOOL_FRACTION or re-seed ES_SEED."
)

# --- imbalance --------------------------------------------------------------
_pos_train = int(y_train.sum())
_neg_train = len(y_train) - _pos_train
SCALE_POS_WEIGHT = _neg_train / max(_pos_train, 1)

# --- base parameters, shared -----------------------------------------------
XGB_BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="aucpr",        # the selection metric for this project
    tree_method="hist",
    max_depth=3,                # shallow: the spec's guard against memorising school quirks
    min_child_weight=20,        # a floor, so a leaf cannot be built on a handful of rows
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    missing=np.nan,             # explicit: NaN is a branch, not a value to fill
)

BORUTA_N_ESTIMATORS = 200       # FIXED - BorutaShap supplies no eval_set
EVAL_N_ESTIMATORS = 2000        # an upper bound; early stopping picks the number
EARLY_STOPPING_ROUNDS = 50


def boruta_model(scale_pos_weight: float) -> XGBClassifier:
    """The estimator BorutaSHAP refits every trial. NO early stopping.

    BorutaShap.Train_model calls model.fit(X, y, sample_weight=..., verbose=False)
    with no eval_set, so an estimator carrying early_stopping_rounds either
    raises or stops on something it was never given.
    """
    m = XGBClassifier(**XGB_BASE_PARAMS,
                      n_estimators=BORUTA_N_ESTIMATORS,
                      scale_pos_weight=scale_pos_weight)
    assert m.get_params().get("early_stopping_rounds") is None, (
        "the BorutaSHAP estimator has early_stopping_rounds set. BorutaShap's "
        "Train_model() supplies no eval_set, so xgboost raises 'Must have at "
        "least 1 validation dataset for early stopping' on the first trial - or "
        "silently stops against the training matrix. Use a fixed n_estimators "
        "here and keep early stopping for eval_model()."
    )
    return m


def eval_model(scale_pos_weight: float = SCALE_POS_WEIGHT) -> XGBClassifier:
    """The evaluation estimator: base parameters + early stopping on AUCPR.

    §11 fits it with eval_set=[(X_es, y_es)] - a school-disjoint half of
    holdout_school. Never the ship gate.
    """
    return XGBClassifier(**XGB_BASE_PARAMS,
                         n_estimators=EVAL_N_ESTIMATORS,
                         early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                         scale_pos_weight=scale_pos_weight)


print(f"train cell            {_pos_train:,} positive / {_neg_train:,} negative "
      f"({y_train.mean():.2%})")
print(f"scale_pos_weight      {SCALE_POS_WEIGHT:.1f}  (train cell's own neg/pos)")
print("                      the spec says TUNE THIS DOWNWARD on AUCPR - "
      "over-weighting positives")
print("                      hurts precision and inflates the probabilities. Not "
      "tuned here.")
print("                      Training only: no evaluation cell is reweighted.")
print(f"\nearly-stopping eval set (from {ES_SPLIT}, school-disjoint from the reported half):")
print(f"  eval    {len(ES_FRAME):>7,} rows  {len(ES_SCHOOLS):>5,} schools  "
      f"{int(y_es.sum()):>5,} positives ({y_es.mean():.2%})")
print(f"  report  {len(HS_READ_FRAME):>7,} rows  {len(READ_SCHOOLS_HS):>5,} schools  "
      f"{int(HS_READ_FRAME[Y_COL].sum()):>5,} positives "
      f"({HS_READ_FRAME[Y_COL].mean():.2%})")
print(f"\nboruta_model  n_estimators={BORUTA_N_ESTIMATORS}, early stopping OFF")
print(f"eval_model    n_estimators<={EVAL_N_ESTIMATORS}, "
      f"early_stopping_rounds={EARLY_STOPPING_ROUNDS} on aucpr")

train cell            6,829 positive / 202,865 negative (3.26%)
scale_pos_weight      29.7  (train cell's own neg/pos)
                      the spec says TUNE THIS DOWNWARD on AUCPR - over-weighting positives
                      hurts precision and inflates the probabilities. Not tuned here.
                      Training only: no evaluation cell is reweighted.

early-stopping eval set (from holdout_school, school-disjoint from the reported half):
  eval     45,098 rows  4,718 schools  1,498 positives (3.32%)
  report   45,109 rows  4,717 schools  1,468 positives (3.25%)

boruta_model  n_estimators=200, early stopping OFF
eval_model    n_estimators<=2000, early_stopping_rounds=50 on aucpr


---

## 9. BorutaSHAP

Runs BorutaSHAP on the `train` cell and nothing else. Every constant is at the top of the cell so the
run can be scaled up without restructuring anything below it:

| constant | what it does |
| --- | --- |
| `BORUTA_N_TRIALS` | trials in the Boruta loop. A modest default for a first pass; raise it and the tentative set shrinks |
| `BORUTA_SAMPLE_N` | target row count for a stratified sample of the `train` cell. **Every positive row is kept** and the negatives are sampled down to hit the target. `None` uses the whole cell |
| `BORUTA_SEED` | the negative sample and BorutaShap's own `random_state` |

Printed before the run: the sampled row count, positive count and positive rate, and the sampled
cell's own `neg/pos` ratio - which is what `scale_pos_weight` is set from for **this** fit, since
down-sampling negatives changes the imbalance the estimator sees. After it: wall-clock time and the
accepted / tentative / rejected counts.

What is passed, against the 1.0.17 signature: `importance_measure="shap"` (lowercase - the `"Shap"`
default raises), `classification=True`, `n_trials`, `random_state`, `sample=False`,
`train_or_test="train"`, `normalize=True`, `verbose=False`. `train_or_test` is BorutaShap's
in-sample / out-of-sample switch for the SHAP computation - `"train"` fits each trial on all of the
sampled rows instead of an internal 70/30 cut, which is what a small positive count needs. It has
nothing to do with this project's `test` cell.

BorutaShap prints `Warning there are missing values in your data !` and carries on: its
`check_missing_values` only tolerates NaN for models whose type name contains `xgb`, which is exactly
why the estimator is XGBoost. Nothing is imputed.

Two things to know about the importance numbers before reading them as importances. With
`normalize=True` they are **z-scored within each trial**, so they are centred near zero and a
negative value is normal - it means "below the average feature in that trial", not "harmful". And a
feature rejected mid-run is dropped from the matrix, so its mean is taken over the trials it
survived: `trials_survived` is reported per feature, and a rank comparison between a
25-trial survivor and a 12-trial casualty is indicative rather than exact. The decision column, not
the level, is the output of this section.

In [50]:
# --- constants: scale the run from here ------------------------------------
BORUTA_N_TRIALS = 100        # modest first pass. 50-100 for a run you will act on.
BORUTA_SAMPLE_N = None    # target rows; None = the whole train cell
BORUTA_SEED = 42

# --- the stratified row sample: every positive, negatives sampled down ------
# Sampling positives away is not an option - they are the scarce thing, and the
# hit statistics are driven by them.
_train_frame = cell(FIT_SPLIT)
_pos_idx = _train_frame.index[_train_frame[Y_COL] == 1]
_neg_idx = _train_frame.index[_train_frame[Y_COL] == 0]

if BORUTA_SAMPLE_N is None or BORUTA_SAMPLE_N >= len(_train_frame):
    BORUTA_IDX = _train_frame.index
    BORUTA_SAMPLED = False
    _neg_target = len(_neg_idx)
else:
    _neg_target = max(BORUTA_SAMPLE_N - len(_pos_idx), 0)
    assert _neg_target > 0, (
        f"BORUTA_SAMPLE_N={BORUTA_SAMPLE_N:,} is smaller than the "
        f"{len(_pos_idx):,} positive rows in the {FIT_SPLIT} cell, and positives "
        "are never sampled away. Raise BORUTA_SAMPLE_N above the positive count "
        "or set it to None."
    )
    _rng = np.random.default_rng(BORUTA_SEED)
    _keep_neg = _rng.choice(np.asarray(_neg_idx), size=min(_neg_target, len(_neg_idx)),
                            replace=False)
    BORUTA_IDX = pd.Index(np.concatenate([np.asarray(_pos_idx), _keep_neg]))
    BORUTA_SAMPLED = True

X_boruta_in = X_train.loc[BORUTA_IDX]
y_boruta_in = y_train.loc[BORUTA_IDX]

assert int(y_boruta_in.sum()) == len(_pos_idx), (
    f"the sample has {int(y_boruta_in.sum()):,} positives but the {FIT_SPLIT} "
    f"cell has {len(_pos_idx):,} - the negative-only down-sample dropped "
    "positive rows."
)
assert list(X_boruta_in.columns) == MODEL_COLS, "the sample is not exactly MODEL_COLS"

_b_pos = int(y_boruta_in.sum())
_b_neg = len(y_boruta_in) - _b_pos
SCALE_POS_WEIGHT_BORUTA = _b_neg / max(_b_pos, 1)

print(f"BORUTA_N_TRIALS  {BORUTA_N_TRIALS}")
print(f"BORUTA_SAMPLE_N  {BORUTA_SAMPLE_N if BORUTA_SAMPLE_N is not None else 'None (whole cell)'}"
      f"   -> {'SAMPLED' if BORUTA_SAMPLED else 'whole cell used'}")
print(f"BORUTA_SEED      {BORUTA_SEED}")
print(f"\nsampled rows     {len(X_boruta_in):,} of {len(_train_frame):,} in the "
      f"{FIT_SPLIT} cell ({len(X_boruta_in) / len(_train_frame):.1%})")
print(f"  positives      {_b_pos:,}  ({y_boruta_in.mean():.2%} of the sample; "
      f"{y_train.mean():.2%} in the full cell)")
print(f"  negatives      {_b_neg:,}")
print(f"  schools        {_train_frame.loc[BORUTA_IDX, 'school_id'].nunique():,}")
print(f"  features       {X_boruta_in.shape[1]}")
print(f"  scale_pos_weight for THIS fit: {SCALE_POS_WEIGHT_BORUTA:.1f} "
      f"(the sample's own neg/pos; the full cell's is {SCALE_POS_WEIGHT:.1f})")

BORUTA_N_TRIALS  100
BORUTA_SAMPLE_N  None (whole cell)   -> whole cell used
BORUTA_SEED      42

sampled rows     209,694 of 209,694 in the train cell (100.0%)
  positives      6,829  (3.26% of the sample; 3.26% in the full cell)
  negatives      202,865
  schools        21,893
  features       177
  scale_pos_weight for THIS fit: 29.7 (the sample's own neg/pos; the full cell's is 29.7)


In [51]:
# --- the run ---------------------------------------------------------------
# One call. `train_or_test="train"` is BorutaShap's SHAP-computation switch, not
# this project's split cells: it fits each trial on all of the sampled rows
# rather than an internal 70/30 cut.
selector = BorutaShap(
    model=boruta_model(SCALE_POS_WEIGHT_BORUTA),
    importance_measure=IMPORTANCE_MEASURE,   # "shap", lowercase
    classification=True,
    percentile=100,
    pvalue=0.05,
)

_t0 = time.perf_counter()
selector.fit(
    X=X_boruta_in,
    y=y_boruta_in,
    n_trials=BORUTA_N_TRIALS,
    random_state=BORUTA_SEED,
    sample=False,            # no internal isolation-forest row sample; §9 above sampled
    train_or_test="train",
    normalize=True,
    verbose=False,
)
BORUTA_SECONDS = time.perf_counter() - _t0

ACCEPTED = sorted(selector.accepted)
TENTATIVE = sorted(map(str, selector.tentative))
REJECTED = sorted(selector.rejected)

assert set(ACCEPTED) | set(TENTATIVE) | set(REJECTED) == set(MODEL_COLS), (
    "accepted + tentative + rejected do not partition MODEL_COLS: "
    f"{len(ACCEPTED)} + {len(TENTATIVE)} + {len(REJECTED)} = "
    f"{len(ACCEPTED) + len(TENTATIVE) + len(REJECTED)} vs {len(MODEL_COLS)} "
    "features. BorutaShap derives all three from the columns it was handed, so a "
    "mismatch means the matrix changed under it."
)

# --- the SHAP importance table ---------------------------------------------
# selector.history_x is one row per trial (row 0 is the zero-initialised row,
# hence .iloc[1:]) plus four shadow columns. A feature rejected mid-run is
# dropped from the matrix, so its later rows are NaN - mean/std skip them, and
# `trials` records how many trials each feature actually survived.
_hist = selector.history_x.iloc[1:]
_shadow_cols = ["Max_Shadow", "Min_Shadow", "Mean_Shadow", "Median_Shadow"]
_feat_hist = _hist.drop(columns=_shadow_cols)

_decision = {**{c: "Accepted" for c in ACCEPTED},
             **{c: "Tentative" for c in TENTATIVE},
             **{c: "Rejected" for c in REJECTED}}

IMPORTANCE = (pd.DataFrame({
    "feature": _feat_hist.columns,
    "mean_shap_importance": _feat_hist.mean(axis=0).to_numpy(),
    "sd_shap_importance": _feat_hist.std(axis=0).to_numpy(),
    "trials_survived": _feat_hist.notna().sum(axis=0).to_numpy(),
})
    .assign(decision=lambda d: d["feature"].map(_decision))
    .sort_values("mean_shap_importance", ascending=False, kind="mergesort")
    .reset_index(drop=True))
IMPORTANCE.insert(0, "rank", np.arange(1, len(IMPORTANCE) + 1))

MEDIAN_MAX_SHADOW = float(_hist["Max_Shadow"].median())

print(f"BorutaSHAP finished in {BORUTA_SECONDS:.1f}s "
      f"({BORUTA_SECONDS / 60:.1f} min) over {BORUTA_N_TRIALS} trials")
print(f"  {len(ACCEPTED):>4} accepted")
print(f"  {len(TENTATIVE):>4} tentative")
print(f"  {len(REJECTED):>4} rejected")
print(f"  {len(MODEL_COLS):>4} candidates in")
print(f"\nmedian Max_Shadow across trials: {MEDIAN_MAX_SHADOW:.5f} "
      "(the bar a feature had to clear)")
print("\ntop 25 by mean SHAP importance:")
print(IMPORTANCE.head(25).to_string(index=False))

if not ACCEPTED:
    print("\n" + "!" * 72)
    print("WARNING: BorutaSHAP accepted NOTHING. At a modest trial count that is")
    print("usually under-powered rather than meaningful - the hit statistics need")
    print("trials before the Bonferroni-corrected p-values can clear 0.05. Raise")
    print("BORUTA_N_TRIALS (and BORUTA_SAMPLE_N) before reading anything into it.")
    print("§11 falls back to accepted + tentative so the comparison still runs.")
    print("!" * 72)

Warning there are missing values in your data !


  0%|          | 0/100 [00:00<?, ?it/s]

BorutaSHAP finished in 421.0s (7.0 min) over 100 trials
    43 accepted
    12 tentative
   122 rejected
   177 candidates in

median Max_Shadow across trials: -0.22421 (the bar a feature had to clear)

top 25 by mean SHAP importance:
 rank                                                  feature  mean_shap_importance  sd_shap_importance  trials_survived decision
    1                                  product_active_students              6.398049            1.398010              100 Accepted
    2                 delta_product_active_students_vs_12m_ago              5.463720            1.260516              100 Accepted
    3                num_students_with_billing_invoice_created              4.575474            1.002347              100 Accepted
    4         delta_messaging_events_per_active_student_vs_t3m              3.971133            0.974790              100 Accepted
    5                      customer_segment__smb_multisite_2_9              2.202820            0.605738      

---

## 10. Persist the selection

Writes the outcome to `data/raw/`, named after `STEM`, so a result can be traced back to the run that
produced it:

- `..._boruta_selection.json` - the three lists, plus the run config: trials, sample size and whether
  sampling actually happened, seeds, the estimator's parameters, `scale_pos_weight`, wall-clock time,
  the source parquet and JSON names, and the library versions.
- `..._boruta_importance.csv` - the SHAP importance table: rank, feature, mean and sd importance,
  trials survived, decision.

Both are read back and checked against the in-memory objects before the summary prints.

In [52]:
SELECTION_JSON = RAW_DIR / f"{STEM}_boruta_selection.json"
IMPORTANCE_CSV = RAW_DIR / f"{STEM}_boruta_importance.csv"

SELECTION_SCHEMA_VERSION = 1

SELECTION = {
    "schema_version": SELECTION_SCHEMA_VERSION,
    "generated_at_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
    "source_encoded_parquet": ENCODED.name,
    "source_encoding_json": ENCODING.name,
    "n_candidates": len(MODEL_COLS),
    "accepted": ACCEPTED,
    "tentative": TENTATIVE,
    "rejected": REJECTED,
    "run_config": {
        "fit_split": FIT_SPLIT,
        "read_splits": list(READ_SPLITS),
        "splits_never_loaded": list(NEVER_LOADED),
        "boruta_n_trials": BORUTA_N_TRIALS,
        "boruta_sample_n": BORUTA_SAMPLE_N,
        "boruta_sampled": bool(BORUTA_SAMPLED),
        "boruta_rows_used": int(len(X_boruta_in)),
        "boruta_positives_used": int(y_boruta_in.sum()),
        "boruta_positive_rate": float(y_boruta_in.mean()),
        "boruta_seed": BORUTA_SEED,
        "random_state": RANDOM_STATE,
        "importance_measure": IMPORTANCE_MEASURE,
        "train_or_test": "train",
        "pvalue": 0.05,
        "percentile": 100,
        "xgb_base_params": {k: (v if not isinstance(v, float) or np.isfinite(v) else str(v))
                            for k, v in XGB_BASE_PARAMS.items()},
        "boruta_n_estimators": BORUTA_N_ESTIMATORS,
        "scale_pos_weight_train_cell": float(SCALE_POS_WEIGHT),
        "scale_pos_weight_used_for_boruta": float(SCALE_POS_WEIGHT_BORUTA),
        "wall_clock_seconds": round(float(BORUTA_SECONDS), 1),
        "median_max_shadow": MEDIAN_MAX_SHADOW,
        "binom_test_shim_installed": bool(BINOM_TEST_SHIM_NEEDED),
    },
    "versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "shap": shap.__version__,
    },
    "notes": (
        "BorutaSHAP over the encoding JSON's model_cols, fitted on the 'train' "
        "cell only. The ship-gate cell is not loaded anywhere in "
        "data/feature_selection.ipynb. Tentative features are neither accepted "
        "nor rejected: raise boruta_n_trials to resolve them. Leakage-risk "
        "features are FLAGGED, never auto-dropped - see the notebook's §13."
    ),
}

with SELECTION_JSON.open("w") as _f:
    json.dump(SELECTION, _f, indent=2)
    _f.write("\n")

IMPORTANCE.to_csv(IMPORTANCE_CSV, index=False)

# --- read both back --------------------------------------------------------
_sel_back = json.loads(SELECTION_JSON.read_text())
assert _sel_back["accepted"] == ACCEPTED, "the JSON's accepted list does not round trip"
assert (_sel_back["run_config"]["boruta_rows_used"] == len(X_boruta_in)
        and _sel_back["source_encoded_parquet"] == ENCODED.name), (
    "the persisted run config does not describe this run, so the artefact cannot "
    "be traced back to it"
)
_imp_back = pd.read_csv(IMPORTANCE_CSV)
assert list(_imp_back["feature"]) == list(IMPORTANCE["feature"]), (
    "the importance CSV's feature order does not round trip - the rank column "
    "would no longer match the rows"
)

print(f"wrote  {SELECTION_JSON.name}   ({SELECTION_JSON.stat().st_size / 1e3:.1f} KB)")
print(f"       {IMPORTANCE_CSV.name}   ({IMPORTANCE_CSV.stat().st_size / 1e3:.1f} KB)")
print(f"beside {ENCODED.name}")
print(f"\nround trip OK: {len(_imp_back)} importance rows, "
      f"{len(_sel_back['accepted'])} accepted / {len(_sel_back['tentative'])} tentative / "
      f"{len(_sel_back['rejected'])} rejected")

wrote  churn_v1_is_reportable_request_202401_202605_h3_boruta_selection.json   (9.7 KB)
       churn_v1_is_reportable_request_202401_202605_h3_boruta_importance.csv   (16.5 KB)
beside churn_v1_is_reportable_request_202401_202605_h3_encoded.parquet

round trip OK: 177 importance rows, 43 accepted / 12 tentative / 122 rejected


---

## 11. Evaluation harness

### 11a. Helpers

`precision@k` and `recall@k` at `k` in `K_VALUES`, plus PR-AUC (average precision) and the base rate.

**`precision@k` is computed PER MONTH and then averaged.** The rep works a monthly list of `k`
schools, so the quantity of interest is "of the `k` schools surfaced this month, how many filed?"
Pooling nine months and taking one global top-`k` answers a different question - it lets a single
month with high scores fill the whole list - so `pooled_at_k` computes that too and both are
reported, side by side, so the gap is visible rather than assumed.

The base rate is reported next to every figure, so `precision@k` reads as a **lift over random**
rather than as a bare number. `recall@k` per month is against that month's own positives; a month
with no positives contributes `NaN` and is excluded from the average, and the count of such months is
printed.

Ties are broken by row order (`kind="mergesort"`), so the same scores give the same list twice.

`per_month_at_k` also returns `hits` - the at-risk schools a month's list actually found. §12 and §14
reuse these two helpers for every ranker they compare, so the model and the baselines are measured by
the same code.

In [53]:
K_VALUES = (40, 45, 50)     # the rep works a list of 40-50 schools a month
K_PRIMARY = 50


def per_month_at_k(frame: pd.DataFrame, k: int) -> pd.DataFrame:
    """Per-month precision@k / recall@k / base rate.

    `frame` needs dim_month, the label and a `score` column. One monthly list of
    k schools per month - the shape of the actual motion.
    """
    rows = []
    for _m, _g in frame.groupby("dim_month", sort=True):
        _top = _g.sort_values("score", ascending=False, kind="mergesort").head(k)
        _hit = int(_top[Y_COL].sum())
        _all = int(_g[Y_COL].sum())
        rows.append({
            "dim_month": _m,
            "rows": len(_g),
            "k_used": len(_top),            # < k in a month with fewer rows than k
            "positives": _all,
            "hits": _hit,                   # at-risk schools the list actually found
            "precision_at_k": _hit / len(_top) if len(_top) else np.nan,
            "recall_at_k": _hit / _all if _all else np.nan,
            "base_rate": _all / len(_g) if len(_g) else np.nan,
        })
    return pd.DataFrame(rows)


def pooled_at_k(frame: pd.DataFrame, k: int) -> dict:
    """One global top-k over every month at once - the WRONG quantity, reported
    for comparison. k schools in total, not k per month."""
    _top = frame.sort_values("score", ascending=False, kind="mergesort").head(k)
    _hit = int(_top[Y_COL].sum())
    _all = int(frame[Y_COL].sum())
    return {
        "pooled_precision_at_k": _hit / len(_top) if len(_top) else np.nan,
        "pooled_recall_at_k": _hit / _all if _all else np.nan,
    }


def score_cell(model, frame: pd.DataFrame, feature_cols) -> pd.DataFrame:
    """Attach `score` to a copy of the identifier/label columns of `frame`.

    Scores only: the frame handed back carries no features, so nothing
    downstream can accidentally fit on an evaluation cell. No sample weighting
    is applied anywhere here - an evaluation cell keeps its true positive rate.
    """
    out = frame[["school_id", "dim_month", Y_COL]].copy()
    out["score"] = model.predict_proba(frame[feature_cols])[:, 1]
    return out


def evaluate(model, frame: pd.DataFrame, feature_cols, label: str,
             feature_set: str) -> tuple:
    """(summary rows, per-month table) for one model on one evaluation cell."""
    scored = score_cell(model, frame, feature_cols)
    ap = float(average_precision_score(scored[Y_COL], scored["score"]))
    base = float(scored[Y_COL].mean())

    summary, per_month = [], {}
    for k in K_VALUES:
        pm = per_month_at_k(scored, k)
        per_month[k] = pm
        pooled = pooled_at_k(scored, k)
        summary.append({
            "cell": label,
            "feature_set": feature_set,
            "n_features": len(feature_cols),
            "rows": len(scored),
            "months": int(pm["dim_month"].nunique()),
            "positives": int(scored[Y_COL].sum()),
            "base_rate": base,
            "k": k,
            "precision_at_k_permonth_mean": float(pm["precision_at_k"].mean(skipna=True)),
            "precision_at_k_permonth_sd": float(pm["precision_at_k"].std(skipna=True)),
            "recall_at_k_permonth_mean": float(pm["recall_at_k"].mean(skipna=True)),
            "lift_over_base": (float(pm["precision_at_k"].mean(skipna=True)) / base
                               if base else np.nan),
            "pr_auc": ap,
            **pooled,
            "months_without_a_positive": int(pm["positives"].eq(0).sum()),
        })
    return summary, per_month


print(f"K_VALUES {K_VALUES}, primary k = {K_PRIMARY}")
print("precision@k is per-month then averaged; pooled_precision_at_k is the single")
print("global top-k over all months, reported alongside so the difference is visible.")

K_VALUES (40, 45, 50), primary k = 50
precision@k is per-month then averaged; pooled_precision_at_k is the single
global top-k over all months, reported alongside so the difference is visible.


### 11b. Full candidate set vs. the BorutaSHAP subset

Fits `eval_model()` twice on the `train` cell - once on all of `MODEL_COLS`, once on the accepted
subset - with the same parameters, the same seed and the same early-stopping eval set, then reads both
on `holdout_school` (the reporting half, school-disjoint from that eval set) and on `holdout_time`.
That is the number behind "did selection help?".

The two cells measure different things and are shown side by side, never averaged together:
`holdout_school` is **new schools, same era**; `holdout_time` is **the same schools, a new era**.
A gap between them is the school effect and the time effect separating.

If BorutaSHAP accepted nothing, the subset falls back to accepted + tentative with a loud note - an
empty feature set cannot be fitted, and reporting the fallback is better than reporting nothing.

Writes `..._selection_eval.csv`.

In [54]:
if ACCEPTED:
    SELECTED_FEATURES = [c for c in MODEL_COLS if c in set(ACCEPTED)]
    SELECTED_LABEL = "boruta_accepted"
else:
    SELECTED_FEATURES = [c for c in MODEL_COLS if c in set(ACCEPTED) | set(TENTATIVE)]
    SELECTED_LABEL = "boruta_accepted_plus_tentative_FALLBACK"
    print("!" * 72)
    print("NOTE: nothing was accepted, so the 'selected' arm below is "
          "accepted + tentative.")
    print("Read it as 'what survived', not as a selection. Raise BORUTA_N_TRIALS.")
    print("!" * 72 + "\n")

assert SELECTED_FEATURES, (
    "the selected feature set is empty - BorutaSHAP rejected every candidate, so "
    "there is nothing to fit. That is a signal about the run (too few trials, or "
    "too few positives in the sample), not about the features."
)
assert set(SELECTED_FEATURES) <= set(MODEL_COLS), "a selected feature is not in MODEL_COLS"

# Order preserved from MODEL_COLS, so the subset model's feature order is stable.
FEATURE_SETS = {
    "full_candidate_set": list(MODEL_COLS),
    SELECTED_LABEL: SELECTED_FEATURES,
}

# The cells this notebook reads. The ship gate is not among them.
READ_FRAMES = {
    f"holdout_school (report half, {len(READ_SCHOOLS_HS)} schools)": HS_READ_FRAME,
    "holdout_time": cell("holdout_time"),
}

_summaries, PER_MONTH = [], {}
MODELS = {}

for _fs_name, _cols in FEATURE_SETS.items():
    _m = eval_model()
    _m.fit(X_train[_cols], y_train,
           eval_set=[(X_es[_cols], y_es)],   # a holdout cell, never the ship gate
           verbose=False)
    MODELS[_fs_name] = _m
    print(f"{_fs_name:<40} {len(_cols):>4} features, "
          f"best_iteration {_m.best_iteration} of {EVAL_N_ESTIMATORS} "
          f"(best aucpr {_m.best_score:.4f} on the early-stopping set)")

    for _cell_name, _frame in READ_FRAMES.items():
        _rows, _pm = evaluate(_m, _frame, _cols, _cell_name, _fs_name)
        _summaries.extend(_rows)
        PER_MONTH[(_fs_name, _cell_name)] = _pm

EVAL = pd.DataFrame(_summaries)

# --- side by side ----------------------------------------------------------
_show = ["cell", "feature_set", "n_features", "k", "base_rate",
         "precision_at_k_permonth_mean", "lift_over_base",
         "recall_at_k_permonth_mean", "pr_auc", "pooled_precision_at_k"]
_fmt = EVAL[_show].copy()
for _c in ("base_rate", "precision_at_k_permonth_mean", "recall_at_k_permonth_mean",
           "pooled_precision_at_k"):
    _fmt[_c] = _fmt[_c].map(lambda v: f"{v:.3%}")
_fmt["lift_over_base"] = _fmt["lift_over_base"].map(lambda v: f"{v:.2f}x")
_fmt["pr_auc"] = _fmt["pr_auc"].map(lambda v: f"{v:.4f}")

print("\n" + "=" * 72)
print("PRECISION@K PER MONTH, AVERAGED  (pooled shown for contrast)")
print("=" * 72)
print(_fmt.to_string(index=False))

print(f"\nper-month precision@{K_PRIMARY} series:")
for (_fs_name, _cell_name), _pm in PER_MONTH.items():
    _s = _pm[K_PRIMARY]
    print(f"\n  {_fs_name}  |  {_cell_name}")
    print("    " + "  ".join(f"{m:%Y-%m}" for m in _s["dim_month"]))
    print("    " + "  ".join(f"{v:>7.1%}" for v in _s["precision_at_k"]))
    print(f"    mean {_s['precision_at_k'].mean(skipna=True):.2%}  "
          f"sd {_s['precision_at_k'].std(skipna=True):.2%}  "
          f"base rate {_s['base_rate'].mean(skipna=True):.2%}  "
          f"months without a positive: {int(_s['positives'].eq(0).sum())}")

# --- did selection help? ---------------------------------------------------
print("\n" + "=" * 72)
print(f"DID SELECTION HELP?  (k = {K_PRIMARY})")
print("=" * 72)
_p = EVAL[EVAL["k"] == K_PRIMARY]
for _cell_name in READ_FRAMES:
    _f = _p[(_p["cell"] == _cell_name) & (_p["feature_set"] == "full_candidate_set")].iloc[0]
    _s = _p[(_p["cell"] == _cell_name) & (_p["feature_set"] == SELECTED_LABEL)].iloc[0]
    print(f"\n{_cell_name}   base rate {_f['base_rate']:.3%}")
    print(f"  {'':<26}{'full':>14}{'selected':>14}{'delta':>12}")
    for _lbl, _key, _pct in (
        (f"precision@{K_PRIMARY} (per month)", "precision_at_k_permonth_mean", True),
        (f"recall@{K_PRIMARY} (per month)", "recall_at_k_permonth_mean", True),
        ("PR-AUC", "pr_auc", False),
        (f"pooled precision@{K_PRIMARY}", "pooled_precision_at_k", True),
    ):
        if _pct:
            print(f"  {_lbl:<26}{_f[_key]:>13.2%}{_s[_key]:>14.2%}"
                  f"{_s[_key] - _f[_key]:>+11.2%}")
        else:
            print(f"  {_lbl:<26}{_f[_key]:>13.4f}{_s[_key]:>14.4f}"
                  f"{_s[_key] - _f[_key]:>+12.4f}")
    print(f"  features                  {int(_f['n_features']):>13}"
          f"{int(_s['n_features']):>14}{int(_s['n_features']) - int(_f['n_features']):>+12}")

EVAL_CSV = RAW_DIR / f"{STEM}_selection_eval.csv"
EVAL.to_csv(EVAL_CSV, index=False)
print(f"\nwrote  {EVAL_CSV.name}   ({EVAL_CSV.stat().st_size / 1e3:.1f} KB)")
print("\nBoth cells are read on a model whose early stopping used a school-disjoint")
print("half of holdout_school. holdout_time is untouched by it, so it is the "
      "cleaner read of the two.")

full_candidate_set                        177 features, best_iteration 123 of 2000 (best aucpr 0.1616 on the early-stopping set)
boruta_accepted                            43 features, best_iteration 70 of 2000 (best aucpr 0.1544 on the early-stopping set)

PRECISION@K PER MONTH, AVERAGED  (pooled shown for contrast)
                                      cell        feature_set  n_features  k base_rate precision_at_k_permonth_mean lift_over_base recall_at_k_permonth_mean pr_auc pooled_precision_at_k
holdout_school (report half, 4717 schools) full_candidate_set         177 40    3.254%                      24.792%          7.62x                    8.037% 0.1551               35.000%
holdout_school (report half, 4717 schools) full_candidate_set         177 45    3.254%                      24.444%          7.51x                    8.939% 0.1551               33.333%
holdout_school (report half, 4717 schools) full_candidate_set         177 50    3.254%                      24.667%        

---

## 12. The `k` sweep and the operating point

§11 answers "did selection help?" at `k` = 40-50. This section answers a different question: what the
precision / recall trade-off looks like across the whole range of list sizes.

`k` in `K_SWEEP`, plus the whole monthly population as the ceiling. Per evaluation cell and per `k`:
`precision@k`, `recall@k`, at-risk schools found per month (`precision@k` x `k`, read straight off
the per-month hit count), and lift over that cell's base rate.

Both cells read here are `holdout_school` (the reporting half) and `holdout_time`. The ship gate is
not among them. Every fit reuses §8's configuration and §11b's already-fitted boosters - nothing here
introduces a second set of hyperparameters. The per-row scores are cached in `SCORED` and §14 reuses
them, so the model in the sweep and the model in the baseline comparison are one fit.

**Computed per calendar month and then averaged, never pooled.** One list of `k` schools per month is
the shape of the motion; a single global top-`k` over nine months lets one month fill the whole list
and answers a question nobody asked. Where `k` exceeds a month's row count the month contributes its
row count instead, and `k_used_mean` reports the effective `k`.

`rank_at_k` takes `skip_short_months`, off here and **on in §14b**: a ranker there scores only the
rows it can score, so a month holding fewer than `k` of them would report precision over whatever it
held and be averaged in against months that got a full list. `months_short_skipped` counts them.

At `k` = the whole month, `precision@k` **is** the base rate, `recall@k` is 1.0 and lift is 1.0 - the
cell asserts all three, because they are the arithmetic identity that catches a broken sweep.

Writes `..._k_sweep.csv`.

In [55]:
# --- the sweep -------------------------------------------------------------
K_SWEEP = (40, 50, 100, 200, 500, 1000, 2000, 5000, 10_000, None)  # None = the whole month
REP_CAPACITY = (40, 50)          # schools a rep can work in a month; §14c reads it


def rank_at_k(scored: pd.DataFrame, k, cell_label: str, ranker: str,
              population: str, skip_short_months: bool = False) -> dict:
    """One summary row: one ranker, one k, per month then averaged.

    k=None means the whole monthly population - the ceiling, where precision is
    the base rate and recall is 1.0. Where k exceeds a month's row count the
    month contributes its row count, so k_used_mean is the effective k actually
    used rather than the requested one.

    skip_short_months=True drops a month with fewer than k scorable rows
    instead, which is what §14b needs: an RHS ranker scores only the rows whose
    trend exists, and a month that cannot fill a list of k would otherwise
    report precision over whatever it held. Ignored at k=None, where the whole
    month IS the list by definition.
    """
    kk = len(scored) if k is None else k
    pm = per_month_at_k(scored, kk)

    n_short = 0
    if skip_short_months and k is not None:
        _short = pm["rows"] < kk
        n_short = int(_short.sum())
        assert not _short.all(), (
            f"on {cell_label!r}, every one of the {len(pm)} month(s) that "
            f"{ranker!r} can score has fewer than k={kk:,} scorable rows (the "
            f"largest holds {int(pm['rows'].max()):,}). There is no month-sized "
            f"list of {kk:,} to measure, so precision@{kk:,} here would silently "
            "be precision over whatever the month happened to hold. Drop this k "
            "from K_BASELINE, or evaluate on a wider cell."
        )
        pm = pm.loc[~_short]

    base = float(pm["base_rate"].mean(skipna=True))
    prec = float(pm["precision_at_k"].mean(skipna=True))
    return {
        "cell": cell_label,
        "ranker": ranker,
        "population": population,
        "k_label": "full month" if k is None else str(k),
        "k_requested": -1 if k is None else int(k),
        "k_used_mean": float(pm["k_used"].mean()),
        "months": int(len(pm)),
        "months_short_skipped": n_short,
        "rows": int(len(scored)),
        "positives": int(scored[Y_COL].sum()),
        "base_rate_permonth_mean": base,
        "base_rate_pooled": float(scored[Y_COL].mean()),
        "precision_at_k": prec,
        "recall_at_k": float(pm["recall_at_k"].mean(skipna=True)),
        "found_per_month": float(pm["hits"].mean()),
        "lift_over_base": prec / base if base else np.nan,
    }


# Scores are computed once here and reused by §14, so the model in the sweep and
# the model in the baseline comparison are the same numbers, not two fits.
SCORED = {}
_sweep_rows = []
for _fs_name, _cols in FEATURE_SETS.items():
    for _cell_name, _frame in READ_FRAMES.items():
        _s = score_cell(MODELS[_fs_name], _frame, _cols)
        SCORED[(_fs_name, _cell_name)] = _s
        for _k in K_SWEEP:
            _sweep_rows.append(rank_at_k(_s, _k, _cell_name, _fs_name, "full_population"))

K_SWEEP_TABLE = pd.DataFrame(_sweep_rows)

# --- the identities at the ceiling -----------------------------------------
# At k = the whole month the list is the population: precision is the base rate,
# recall is 1, lift is 1. If the sweep is subtly wrong these stop holding.
_ceiling = K_SWEEP_TABLE[K_SWEEP_TABLE["k_requested"] == -1]
assert len(_ceiling) == len(FEATURE_SETS) * len(READ_FRAMES), (
    f"{len(_ceiling)} whole-month row(s), expected "
    f"{len(FEATURE_SETS) * len(READ_FRAMES)} - the sweep did not run the ceiling "
    "for every model x cell."
)
assert np.allclose(_ceiling["precision_at_k"], _ceiling["base_rate_permonth_mean"]), (
    "precision at k = the whole month is not the per-month base rate:\n"
    + _ceiling[["cell", "ranker", "precision_at_k", "base_rate_permonth_mean"]].to_string(index=False)
    + "\n  Taking the whole population as the list must reproduce the base rate "
      "exactly. It does not, so per_month_at_k is not selecting what it reports."
)
assert np.allclose(_ceiling["recall_at_k"], 1.0), (
    "recall at k = the whole month is not 1.0: "
    f"{_ceiling['recall_at_k'].tolist()}. Every positive is in the list by "
    "construction, so anything else means recall is measured against the wrong "
    "denominator."
)
assert np.allclose(_ceiling["lift_over_base"], 1.0), "lift at the ceiling is not 1.0x"

# --- is the reported number really the per-month average? ------------------
# Recomputed by hand, without the helpers, on one (cell, k) case. The pooled
# figure is computed alongside: it is a DIFFERENT quantity and is not what any
# precision_at_k column carries.
_K_PROBE = 200
_probe = SCORED[(SELECTED_LABEL, "holdout_time")]
_by_hand = []
for _m, _g in _probe.groupby("dim_month", sort=True):
    _t = _g.sort_values("score", ascending=False, kind="mergesort").head(_K_PROBE)
    _by_hand.append(int(_t[Y_COL].sum()) / len(_t))
_by_hand_mean = float(np.mean(_by_hand))
_reported = float(K_SWEEP_TABLE.loc[
    (K_SWEEP_TABLE["cell"] == "holdout_time")
    & (K_SWEEP_TABLE["ranker"] == SELECTED_LABEL)
    & (K_SWEEP_TABLE["k_requested"] == _K_PROBE), "precision_at_k"].iloc[0])
_pooled_probe = pooled_at_k(_probe, _K_PROBE)["pooled_precision_at_k"]

assert np.isclose(_reported, _by_hand_mean), (
    f"the reported precision@{_K_PROBE} on holdout_time is {_reported:.6f} but a "
    f"per-month recomputation gives {_by_hand_mean:.6f} (pooled would be "
    f"{_pooled_probe:.6f}). The sweep is not reporting the per-month average it "
    "claims to."
)
assert len(_by_hand) == int(_probe["dim_month"].nunique()), (
    "the per-month recomputation covered "
    f"{len(_by_hand)} month(s) but the cell has {_probe['dim_month'].nunique()} - "
    "months are being dropped before the average."
)

# --- print -----------------------------------------------------------------
_cols_show = ["k_label", "k_used_mean", "base_rate_permonth_mean", "precision_at_k",
              "lift_over_base", "recall_at_k", "found_per_month"]
for _cell_name in READ_FRAMES:
    for _fs_name in FEATURE_SETS:
        _t = K_SWEEP_TABLE[(K_SWEEP_TABLE["cell"] == _cell_name)
                           & (K_SWEEP_TABLE["ranker"] == _fs_name)][_cols_show].copy()
        print("\n" + "=" * 78)
        print(f"K SWEEP  |  {_cell_name}  |  {_fs_name}")
        print("=" * 78)
        print(f"{'k':>11}{'k used':>10}{'base':>9}{'prec@k':>9}{'lift':>8}"
              f"{'recall@k':>10}{'found/mo':>10}")
        for _, _r in _t.iterrows():
            print(f"{_r['k_label']:>11}{_r['k_used_mean']:>10,.0f}"
                  f"{_r['base_rate_permonth_mean']:>9.2%}{_r['precision_at_k']:>9.2%}"
                  f"{_r['lift_over_base']:>7.2f}x{_r['recall_at_k']:>10.2%}"
                  f"{_r['found_per_month']:>10,.1f}")

print("\nper month, then averaged - never one pooled top-k over every month at once.")
print(f"a rep works {REP_CAPACITY[0]}-{REP_CAPACITY[1]} schools a month; every k above that is a "
      "statement about the model, not about a workable list.")

K_SWEEP_CSV = RAW_DIR / f"{STEM}_k_sweep.csv"
K_SWEEP_TABLE.to_csv(K_SWEEP_CSV, index=False)
print(f"\nwrote  {K_SWEEP_CSV.name}   ({K_SWEEP_CSV.stat().st_size / 1e3:.1f} KB)")


K SWEEP  |  holdout_school (report half, 4717 schools)  |  full_candidate_set
          k    k used     base   prec@k    lift  recall@k  found/mo
         40        40    3.27%   24.79%   7.58x     8.04%       9.9
         50        50    3.27%   24.67%   7.55x     9.97%      12.3
        100       100    3.27%   23.92%   7.32x    19.40%      23.9
        200       200    3.27%   19.67%   6.02x    32.29%      39.3
        500       500    3.27%   13.23%   4.05x    54.62%      66.2
       1000     1,000    3.27%    8.95%   2.74x    73.42%      89.5
       2000     2,000    3.27%    5.45%   1.67x    89.21%     109.1
       5000     3,759    3.27%    3.27%   1.00x   100.00%     122.3
      10000     3,759    3.27%    3.27%   1.00x   100.00%     122.3
 full month     3,759    3.27%    3.27%   1.00x   100.00%     122.3

K SWEEP  |  holdout_school (report half, 4717 schools)  |  boruta_accepted
          k    k used     base   prec@k    lift  recall@k  found/mo
         40        40    3.27

---

## 13. Leakage / business-sense surface

Where the features the spec flags as leakage risks landed in the SHAP importances. They describe
things that happen *around* a cancellation conversation, so a high rank is a question for a human,
not an automatic drop:

- `negative_reason_count` and its four `reason_school_*_count` specifics - billing-plan stoppage
  reasons, which a school often sets *while* deciding to leave;
- `months_since_last_cancel_request` and `running_saas_cancel_request_count` - built from the same
  case table the label comes from;
- every `trailing_3m_` / `delta_*_vs_t3m` / `delta_*_vs_12m_ago` variant of the above that the
  extract generated.

Prints rank, mean SHAP importance and decision for each, plus how many of the top 20 features are
flagged. **Nothing is dropped here.** Flagged names that are not in `MODEL_COLS` are listed too - the
extract only keeps variants the warehouse actually carries.

The importance column is §9's z-scored mean, so read the **rank and the decision**, not the level.

In [56]:
# Base signals the spec flags, and the variant naming the extract generates.
LEAKAGE_RISK_BASE = [
    "negative_reason_count",
    "reason_school_temp_paused_count",
    "reason_school_temp_reducing_enrollment_count",
    "reason_school_permanently_closing_count",
    "reason_school_no_longer_using_bw_count",
    "months_since_last_cancel_request",
    "running_saas_cancel_request_count",
]
VARIANT_TEMPLATES = ["{c}", "trailing_3m_{c}", "delta_{c}_vs_t3m", "delta_{c}_vs_12m_ago"]

_flagged = [t.format(c=b) for b in LEAKAGE_RISK_BASE for t in VARIANT_TEMPLATES]
LEAKAGE_RISK_FEATURES = [c for c in _flagged if c in set(MODEL_COLS)]
_absent = [c for c in _flagged if c not in set(MODEL_COLS)]

_rank_of = dict(zip(IMPORTANCE["feature"], IMPORTANCE["rank"]))
LEAKAGE_REPORT = (IMPORTANCE[IMPORTANCE["feature"].isin(LEAKAGE_RISK_FEATURES)]
                  [["rank", "feature", "mean_shap_importance", "decision"]]
                  .reset_index(drop=True))

print(f"{len(LEAKAGE_RISK_FEATURES)} of {len(_flagged)} flagged names are in "
      f"MODEL_COLS ({len(MODEL_COLS)} features):\n")
print(LEAKAGE_REPORT.to_string(index=False) if len(LEAKAGE_REPORT)
      else "  none of the flagged features is in this matrix")

_in_top20 = LEAKAGE_REPORT[LEAKAGE_REPORT["rank"] <= 20]
_accepted_flagged = LEAKAGE_REPORT[LEAKAGE_REPORT["decision"] == "Accepted"]

print(f"\nflagged features in the top 20 by SHAP importance: {len(_in_top20)}")
for _, _r in _in_top20.iterrows():
    print(f"    rank {int(_r['rank']):>3}  {_r['feature']:<52} "
          f"{_r['mean_shap_importance']:>10.5f}  {_r['decision']}")
print(f"flagged features ACCEPTED by BorutaSHAP: {len(_accepted_flagged)} of "
      f"{len(LEAKAGE_RISK_FEATURES)}")

if _absent:
    print(f"\nflagged names NOT in this matrix ({len(_absent)}) - the extract only keeps "
          "variants the warehouse carries:")
    for _c in _absent:
        print(f"    {_c}")

print("\n" + "-" * 72)
print("NOT auto-dropped. A high rank here is a question for a human: does the")
print("feature describe the school's state BEFORE the decision, or the paperwork")
print("AROUND it? Record the call in the vault page, then re-run with the drop")
print("applied to the extract - not with a filter in this notebook.")
print("-" * 72)

22 of 28 flagged names are in MODEL_COLS (177 features):

 rank                                                       feature  mean_shap_importance  decision
   12                              months_since_last_cancel_request              0.563285  Accepted
   17                             running_saas_cancel_request_count              0.403673  Accepted
   28                               reason_school_temp_paused_count              0.134783  Accepted
   39            trailing_3m_reason_school_no_longer_using_bw_count             -0.076591  Rejected
   57                                         negative_reason_count             -0.134218  Rejected
   97                        reason_school_no_longer_using_bw_count             -0.235593 Tentative
  110                  reason_school_temp_reducing_enrollment_count             -0.248747  Rejected
  122                            delta_negative_reason_count_vs_t3m             -0.257630  Rejected
  127                             trailing

---

## 14. Baseline comparison: model vs the Retention Health Score

The project's ship gate: the model must beat the **Retention Health Score** on the same backtest, both
scored on identical school-months. `product_score` is that score, carried through the extract as a
passthrough. It is **never a feature** - §7 asserts it against `MODEL_COLS`, and §14a re-asserts it
against the two feature lists §11b actually fitted.

**The score.** A 24-point composite of eight product signals:

| points | signals |
| --- | --- |
| 4 if active, 2 if any | attendance, messaging, online payment, activity |
| 2 if active, 1 if any | timecard, form, admission funnel, EC lesson plan |

Buckets: `>= 18` **High**, `>= 12` **Medium**, `>= 0` **Low**.

- **14a** rebuilds the score's three-month trend from `product_score`'s own per-school history.
- **14b** ranks three lead lists - the model, worst score first, biggest drop first - at `k` = 40, 50,
  200, 500 and 1000.
- **14c** runs the team's documented trigger rule as the fixed set it actually is, and puts the model
  at that same list size.

The cells read here are `holdout_school` (the reporting half) and `holdout_time`. §12's cached `SCORED`
supplies the model's probabilities, so nothing here fits a second time. The ship gate is not among
them.

### 14a. Rebuilding the trend from `product_score`'s history

`trailing_3m_product_score` and `delta_product_score_vs_t3m` are excluded from the extract as
health-score derivatives, so this cell reconstructs them to the warehouse definition:

| column | definition |
| --- | --- |
| `trailing_3m_product_score` | mean of the school's **previous three months** - the current month is EXCLUDED. Sort by school then month, group by school, `shift(1)`, `rolling(3, min_periods=3).mean()` |
| `delta_product_score_vs_t3m` | `product_score / trailing_3m_product_score - 1`, **NaN when the baseline is 0** (the warehouse's `nullif`) |

Both names are the warehouse's own, and the cell asserts neither is already a column of the parquet -
if a re-run of the extract starts carrying them, read those instead of rebuilding here.

Reports `product_score`'s null rate and range and asserts the range against the documented 24-point
maximum. A value above it means the formula has changed and every bucket above is wrong.

Both columns are computed on the **full frame before any subsetting** - computed inside an evaluation
cell they would be null for that cell's first three months for no reason. Only `product_score` and
the two keys are read, no feature and no label, and the window looks strictly backwards, so the cell
also asserts that dropping the ship-gate rows entirely changes not one value on any row evaluated
below.

The arithmetic is checked against a constructed example whose trailing averages are computable by
hand, including a school with fewer than three prior months and a school whose baseline is zero. The
example is shuffled before being passed in, so the sort is under test too.

**The coverage gap is a finding, not a footnote.** The trend needs three prior months and a non-zero
baseline, so the incumbent rule cannot score every row - roughly a fifth of them on the real extract.
The cell prints the computable share per cell and splits the shortfall into its two causes. The model
scores all of them.

In [57]:
# --- product_score is a passthrough, and this section is the reason why -----
RHS_SCORE_COL = "product_score"
RHS_MAX_POINTS = 24
RHS_BUCKETS = ((18, "High"), (12, "Medium"), (0, "Low"))

assert RHS_SCORE_COL in enc.columns, (
    f"{RHS_SCORE_COL!r} is not a column of {ENCODED.name}, so the incumbent "
    "Retention Health Score cannot be rebuilt and the model has nothing to be "
    "scored against. The extract carries it as a passthrough - re-run "
    "data/dataset_creation.ipynb."
)
assert RHS_SCORE_COL in set(PASSTHROUGH), (
    f"{RHS_SCORE_COL!r} is not in the encoding JSON's passthrough list "
    f"({PASSTHROUGH}). This section scores the model against it, so it has to be "
    "carried in the frame - and carried as a passthrough, not as a feature."
)
assert RHS_SCORE_COL not in set(MODEL_COLS), (
    f"{RHS_SCORE_COL!r} is in MODEL_COLS. The whole point of this section is that "
    "the model is scored AGAINST the Retention Health Score; a model that has the "
    "score as an input beats it trivially and the comparison means nothing."
)
for _fs_name, _cols in FEATURE_SETS.items():
    assert RHS_SCORE_COL not in set(_cols), (
        f"{RHS_SCORE_COL!r} is in the {_fs_name!r} feature list that §11b fitted "
        f"({len(_cols)} features). Checking MODEL_COLS is not enough - what "
        "matters is what the booster was handed. The comparison below is void "
        "until the feature list is fixed upstream."
    )

# --- what the column actually holds -----------------------------------------
_ps = enc[RHS_SCORE_COL].astype("float64")
RHS_NULL_RATE = float(_ps.isna().mean())

assert _ps.notna().any(), (
    f"{RHS_SCORE_COL!r} is null on every one of the {len(enc):,} rows, so the "
    "baseline is empty and there is nothing to compare against."
)
assert float(_ps.max()) <= RHS_MAX_POINTS, (
    f"{RHS_SCORE_COL} reaches {float(_ps.max()):.1f}, above the documented "
    f"{RHS_MAX_POINTS}-point maximum of the Retention Health Score (4+4+4+4 on "
    "attendance / messaging / online payment / activity, 2+2+2+2 on timecard / "
    "form / admission funnel / EC lesson plan). Either the formula changed "
    "upstream - in which case the buckets "
    + ", ".join(f">= {_p} {_n}" for _p, _n in RHS_BUCKETS)
    + " and everything in this section are wrong - or the column is not the score "
      "it is named after. Settle it against the warehouse before reading any "
      "number below."
)
assert float(_ps.min()) >= 0, (
    f"{RHS_SCORE_COL} goes down to {float(_ps.min()):.1f}. The score is a sum of "
    "non-negative point awards, so it cannot be negative; a negative value means "
    "a sentinel is being carried as a measurement and would sit at the top of "
    "the worst-score-first lead list."
)

print(f"Retention Health Score: {RHS_MAX_POINTS}-point composite, buckets "
      + ", ".join(f">= {_p} {_n}" for _p, _n in RHS_BUCKETS))
print(f"{RHS_SCORE_COL}: {RHS_NULL_RATE:.2%} null, range "
      f"{float(_ps.min()):.0f} -> {float(_ps.max()):.0f} of {RHS_MAX_POINTS}, "
      f"median {float(_ps.median()):.0f}, mean {float(_ps.mean()):.1f}")

# --- the two trend columns the extract deliberately does not carry ----------
RHS_T3M_COL = "trailing_3m_product_score"      # the warehouse's own names
RHS_DELTA_COL = "delta_product_score_vs_t3m"

_already_there = [c for c in (RHS_T3M_COL, RHS_DELTA_COL) if c in enc.columns]
assert not _already_there, (
    f"{_already_there} already exist(s) in {ENCODED.name}. The extract "
    "deliberately excludes the Retention Health Score's trend columns, which is "
    "why this cell rebuilds them; if a re-run now carries them, read those "
    "instead of recomputing - two definitions of the same lead-list rule is one "
    "too many."
)


def rhs_trend(frame: pd.DataFrame) -> pd.DataFrame:
    """Rebuild the warehouse's two Retention Health Score trend columns.

    trailing_3m_product_score is the mean of the school's PREVIOUS THREE MONTHS
    - the current month is excluded, which is what the shift(1) before the
    rolling window is for. min_periods=3 means a school with fewer than three
    prior months gets NaN rather than a short average.

    delta_product_score_vs_t3m is product_score / t3m - 1, and NaN when the
    baseline is 0 (the warehouse divides by a nullif, and a ratio against a zero
    baseline is not a number a lead list can be sorted by).

    Sorts by school then month internally, and shifts WITHIN each school, so the
    caller's row order does not matter; the result is returned aligned to
    `frame.index`.
    """
    f = (frame[["school_id", "dim_month", RHS_SCORE_COL]]
         .sort_values(["school_id", "dim_month"], kind="mergesort"))
    score = f[RHS_SCORE_COL].astype("float64")

    prev = score.groupby(f["school_id"], observed=True).shift(1)
    t3m = (prev.groupby(f["school_id"], observed=True)
               .rolling(3, min_periods=3).mean()
               .reset_index(level=0, drop=True))

    base = t3m.where(t3m != 0.0)          # nullif(t3m, 0)
    delta = score / base - 1.0

    return pd.DataFrame({RHS_T3M_COL: t3m.reindex(frame.index),
                         RHS_DELTA_COL: delta.reindex(frame.index)},
                        index=frame.index)


# --- the arithmetic, on an example that can be checked by hand -------------
# school 1: 0, 6, 12, 18, 24, 24 over six consecutive months
#   t3m(month 4) = (0 + 6 + 12) / 3 = 6      <- previous three, NOT (6+12+18)/3 = 12
#   t3m(month 5) = (6 + 12 + 18) / 3 = 12
#   t3m(month 6) = (12 + 18 + 24) / 3 = 18
#   delta(month 4) = 18 / 6 - 1 = 2.0, delta(month 5) = 24 / 12 - 1 = 1.0
# school 2: three months only -> never three PRIOR months -> all NaN
# school 3: 0, 0, 0, 5 -> t3m(month 4) = 0 -> delta is NaN, not inf
_demo = pd.DataFrame({
    "school_id": [1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3],
    "dim_month": pd.to_datetime(
        ["2024-01-01", "2024-02-01", "2024-03-01", "2024-04-01", "2024-05-01", "2024-06-01",
         "2024-01-01", "2024-02-01", "2024-03-01",
         "2024-01-01", "2024-02-01", "2024-03-01", "2024-04-01"]),
    RHS_SCORE_COL: [0.0, 6.0, 12.0, 18.0, 24.0, 24.0,
                    10.0, 20.0, 24.0,
                    0.0, 0.0, 0.0, 5.0],
})
# Shuffled in, so the function's own sort is under test too.
_demo_out = rhs_trend(_demo.sample(frac=1.0, random_state=0)).reindex(_demo.index)
_dt = _demo_out[RHS_T3M_COL].to_numpy()
_dd = _demo_out[RHS_DELTA_COL].to_numpy()

assert np.isnan(_dt[:3]).all(), (
    f"the first three months of a school have a trailing average ({_dt[:3]}). "
    "min_periods=3 over a shifted series cannot produce one - fewer than three "
    "PRIOR months exist."
)
assert _dt[3] == 6.0, (
    f"t3m for 2024-04 is {_dt[3]!r}, expected 6.0 = (0 + 6 + 12) / 3, the three "
    "months BEFORE it. 12.0 = (6 + 12 + 18) / 3 would mean the window includes "
    "the current month, i.e. the shift(1) is missing - which leaks the row's own "
    "score into its own baseline and makes delta_product_score_vs_t3m a "
    "different, tamer quantity than the warehouse's."
)
assert _dt[4] == 12.0 and _dt[5] == 18.0, f"t3m marched wrong: {_dt[3:6]}"
assert np.isclose(_dd[3], 2.0) and np.isclose(_dd[4], 1.0), (
    f"delta is {_dd[3:5]}, expected [2.0, 1.0] from 18/6 - 1 and 24/12 - 1."
)
assert np.isnan(_dd[:3]).all(), "delta exists where the baseline does not"
assert np.isnan(_dt[6:9]).all(), (
    f"a school with three rows total has a trailing average: {_dt[6:9]}. It has "
    "at most two PRIOR months."
)
assert _dt[12] == 0.0 and np.isnan(_dd[12]), (
    f"a zero baseline gave t3m={_dt[12]!r}, delta={_dd[12]!r}. The warehouse "
    "divides by nullif(t3m, 0), so delta must be NaN here - not inf, and not a "
    "silently large number that would sit at the top of a lead list."
)
print("\ntrailing-average arithmetic checked on a constructed example:")
print(pd.concat([_demo, _demo_out], axis=1).to_string(index=False))

# --- rebuilt on the FULL frame, before any subsetting ----------------------
# product_score and the two keys only: no feature and no label of any row is
# read here. The window looks strictly backwards, so the check below shows the
# ship-gate rows contribute nothing to any row this notebook evaluates.
RHS_TREND = rhs_trend(enc)
assert list(RHS_TREND.index) == list(enc.index), "the rebuilt trend is not aligned to the frame"

_trend_without_gate = rhs_trend(enc.loc[enc["split"] != SHIP_GATE_SPLIT])
for _cell_name, _frame in READ_FRAMES.items():
    _a = RHS_TREND.loc[_frame.index]
    _b = _trend_without_gate.loc[_frame.index]
    assert _a.equals(_b), (
        f"the rebuilt trend on {_cell_name} changes when the "
        f"{SHIP_GATE_SPLIT!r} rows are removed from the frame it is computed "
        "over. The trailing window is supposed to look strictly backwards within "
        "a school, and the ship gate is a later era for schools this notebook "
        "does not train on - so it must contribute nothing here. It does, so the "
        "window is not causal."
    )
print(f"\nrebuilt on all {len(enc):,} rows; removing the {SHIP_GATE_SPLIT!r} rows changes "
      "no value on any row evaluated below  OK")

# --- coverage: what fraction of rows the incumbent rule can score at all ----
# Split into its two causes, because they are different problems: no three
# months of prior history, or a baseline of exactly zero.
_t3m_all, _delta_all = RHS_TREND[RHS_T3M_COL], RHS_TREND[RHS_DELTA_COL]
RHS_TREND_COVERAGE = float(_delta_all.notna().mean())

_cov_rows = []
for _label, _idx in [("ALL ROWS", enc.index)] + [(c, f.index) for c, f in READ_FRAMES.items()]:
    _t, _d = _t3m_all.loc[_idx], _delta_all.loc[_idx]
    _cov_rows.append({
        "population": _label,
        "rows": len(_idx),
        "trend_computable": int(_d.notna().sum()),
        "computable_share": float(_d.notna().mean()),
        "no_3m_history": int(_t.isna().sum()),
        "zero_baseline": int((_t == 0.0).sum()),
    })
RHS_COVERAGE_GAP = pd.DataFrame(_cov_rows)
assert (RHS_COVERAGE_GAP["trend_computable"]
        + RHS_COVERAGE_GAP["no_3m_history"]
        + RHS_COVERAGE_GAP["zero_baseline"] == RHS_COVERAGE_GAP["rows"]).all(), (
    "computable + no-history + zero-baseline does not add up to the row count:\n"
    + RHS_COVERAGE_GAP.to_string(index=False)
    + "\n  Those three are supposed to partition every row - a null delta is "
      "either a null baseline or a zero one, and nothing else. They do not, so "
      "the rebuilt delta is null for some third reason."
)

print("\n" + "=" * 78)
print("COVERAGE GAP - WHAT THE INCUMBENT RULE CANNOT SCORE AT ALL")
print("=" * 78)
print(f"  {'population':<44}{'rows':>10}{'computable':>12}{'share':>8}")
for _, _r in RHS_COVERAGE_GAP.iterrows():
    print(f"  {_r['population']:<44}{int(_r['rows']):>10,}"
          f"{int(_r['trend_computable']):>12,}{_r['computable_share']:>8.1%}")
print("\n  the shortfall, by cause:")
for _, _r in RHS_COVERAGE_GAP.iterrows():
    print(f"  {_r['population']:<44}"
          f"{int(_r['no_3m_history']):>10,} no 3m history"
          f"{int(_r['zero_baseline']):>10,} zero baseline")
print(f"\n  -> the trend is computable on {RHS_TREND_COVERAGE:.1%} of all rows. On the rest the")
print("     incumbent rule cannot produce a number at all, so those schools are")
print("     invisible to it. The model scores 100% of them. That gap is a finding")
print("     in its own right, not a caveat on the numbers below.")

Retention Health Score: 24-point composite, buckets >= 18 High, >= 12 Medium, >= 0 Low
product_score: 0.00% null, range 0 -> 24 of 24, median 14, mean 13.1

trailing-average arithmetic checked on a constructed example:
 school_id  dim_month  product_score  trailing_3m_product_score  delta_product_score_vs_t3m
         1 2024-01-01            0.0                        NaN                         NaN
         1 2024-02-01            6.0                        NaN                         NaN
         1 2024-03-01           12.0                        NaN                         NaN
         1 2024-04-01           18.0                        6.0                    2.000000
         1 2024-05-01           24.0                       12.0                    1.000000
         1 2024-06-01           24.0                       18.0                    0.333333
         2 2024-01-01           10.0                        NaN                         NaN
         2 2024-02-01           20.0         

### 14b. Three rankers at matched list sizes

Three lead lists over the same evaluation cell, at `k` in `K_BASELINE`:

| ranker | order |
| --- | --- |
| `model` | the evaluation model's predicted probability, descending |
| `RHS lowest score first` | `product_score` ascending - worst health, highest risk |
| `RHS biggest drop first` | the rebuilt delta ascending - biggest drop, highest risk |

`precision@k` and `recall@k` are computed **per calendar month and then averaged**. The rep works one
monthly list, so a single pooled top-`k` over every month at once is a different quantity and the
wrong one. The base rate and the lift over it are reported next to every figure.

A ranker does not rank a row it cannot score, so the null rows are dropped **for that ranker only** -
the delta is the one with nulls - and `rows`, `months` and `k_used_mean` report the pool actually
used, so a smaller pool shows up in the table instead of quietly changing the comparison. A month
holding fewer than `k` scorable rows is skipped rather than averaged in over a short list
(`skip_short_months=True`); months with nothing scorable never enter the average and are counted in
the coverage line.

The two RHS rankers agree at small `k` on the rows the delta can score, and the cell asserts why:
`product_score` is an integer 0-24, so inside the delta's own subset a row scoring 0 has
`delta = 0 / t3m - 1 = -1.0` exactly, and nothing is lower than either. That tied floor is hundreds of
school-months a month, so below it both rankings are one arbitrary draw from the same block. Its size
and its own precision are printed - the same number without the tie-break.

In [58]:
# --- the three rankers ------------------------------------------------------
K_BASELINE = (40, 50, 200, 500, 1000)
K_BASELINE_HEADLINE = 40

HEADLINE_MODEL = SELECTED_LABEL          # the arm §11b selected, not the full set

MODEL_RANKER = "model"
RHS_SCORE_RANKER = "rhs_lowest_score_first"
RHS_DELTA_RANKER = "rhs_biggest_drop_first"
BASELINE_RANKERS = (MODEL_RANKER, RHS_SCORE_RANKER, RHS_DELTA_RANKER)
RANKER_LABEL = {
    MODEL_RANKER: "model",
    RHS_SCORE_RANKER: "RHS lowest score first",
    RHS_DELTA_RANKER: "RHS biggest drop first",
}
BASELINE_POPULATION = "scorable_by_this_ranker"


def baseline_board(cell_name: str, frame: pd.DataFrame) -> pd.DataFrame:
    """Keys, label, the RHS columns and one ranking score per ranker.

    No features: this frame is for ranking, so nothing downstream can fit on it.
    The two RHS scores are NEGATED, because per_month_at_k takes the highest
    score first and the risky end of both columns is the low end.
    """
    b = frame[["school_id", "dim_month", Y_COL]].copy()
    b[RHS_SCORE_COL] = frame[RHS_SCORE_COL].astype("float64")
    b["rhs_t3m"] = RHS_TREND.loc[frame.index, RHS_T3M_COL]
    b["rhs_delta"] = RHS_TREND.loc[frame.index, RHS_DELTA_COL]

    _s = SCORED[(HEADLINE_MODEL, cell_name)]
    assert list(_s.index) == list(frame.index), (
        f"the cached {HEADLINE_MODEL!r} scores for {cell_name!r} are not aligned "
        "to the frame - §12 scored a different row order, so the model's "
        "probability would be attached to the wrong school-month."
    )
    b[f"score__{MODEL_RANKER}"] = _s["score"]
    b[f"score__{RHS_SCORE_RANKER}"] = -b[RHS_SCORE_COL]
    b[f"score__{RHS_DELTA_RANKER}"] = -b["rhs_delta"]
    return b


BOARDS, _cov, _rows = {}, [], []

for _cell_name, _frame in READ_FRAMES.items():
    _b = baseline_board(_cell_name, _frame)
    BOARDS[_cell_name] = _b
    _months_in_cell = int(_b["dim_month"].nunique())

    for _r in BASELINE_RANKERS:
        _ok = _b[f"score__{_r}"].notna()
        _scored_r = (_b.loc[_ok, ["school_id", "dim_month", Y_COL]]
                       .assign(score=_b.loc[_ok, f"score__{_r}"]))
        _cov.append({
            "cell": _cell_name,
            "ranker_key": _r,
            "ranker": RANKER_LABEL[_r],
            "rows_in_cell": int(len(_b)),
            "scorable": int(_ok.sum()),
            "scorable_share": float(_ok.mean()),
            "months_in_cell": _months_in_cell,
            "months_with_any_scorable": int(_scored_r["dim_month"].nunique()),
        })
        assert len(_scored_r), (
            f"{RANKER_LABEL[_r]!r} can score none of the {len(_b):,} rows of "
            f"{_cell_name!r}, so it cannot be compared against anything. For the "
            "RHS trend that means no school in the cell has three months of prior "
            f"{RHS_SCORE_COL} history - check §14a's coverage table."
        )
        for _k in K_BASELINE:
            _row = rank_at_k(_scored_r, _k, _cell_name, RANKER_LABEL[_r],
                             BASELINE_POPULATION, skip_short_months=True)
            _row["ranker_key"] = _r
            _rows.append(_row)

BASELINE = pd.DataFrame(_rows)
BASELINE_COVERAGE = pd.DataFrame(_cov)

# --- the tied floor, which is why the two RHS columns agree at small k ------
# Inside the delta's own scorable subset the baseline is non-null and non-zero,
# so product_score == 0 gives delta = 0 / t3m - 1 = -1.0 exactly and the two
# risky ends are the same block of rows. Asserted rather than explained away,
# then measured: the floor's own precision is the same number without the
# arbitrary tie-break.
RHS_FLOOR = []
for _cell_name in READ_FRAMES:
    _b = BOARDS[_cell_name]
    _has_delta = _b["rhs_delta"].notna()
    _z = _has_delta & (_b[RHS_SCORE_COL] == 0)
    _d1 = _has_delta & (_b["rhs_delta"] == -1.0)
    assert _z.equals(_d1), (
        f"on {_cell_name!r} the delta-scorable rows with {RHS_SCORE_COL} == 0 "
        f"({int(_z.sum()):,}) are not the rows with delta == -1.0 "
        f"({int(_d1.sum()):,}). Where the delta exists the baseline is non-null "
        "and non-zero, so 0 / t3m - 1 is exactly -1.0 and the two sets must "
        "coincide. They do not, so the rebuilt delta is not the ratio it claims "
        "to be."
    )
    _fm = _b.loc[_z].groupby("dim_month", sort=True).agg(
        floor_rows=(Y_COL, "size"), floor_hits=(Y_COL, "sum"))
    _all_m = _b.loc[_has_delta].groupby("dim_month", sort=True)[Y_COL].agg(["size", "sum"])
    _fm = _fm.reindex(_all_m.index).fillna(0)
    RHS_FLOOR.append({
        "cell": _cell_name,
        "floor_rows_per_month": float(_fm["floor_rows"].mean()),
        "floor_precision": float((_fm["floor_hits"] / _fm["floor_rows"]).mean(skipna=True)),
        "floor_recall": float((_fm["floor_hits"] / _all_m["sum"]).mean(skipna=True)),
    })

# --- the comparison, per cell ----------------------------------------------
_k_labels = [str(_k) for _k in K_BASELINE]
_W = 24

for _cell_name in READ_FRAMES:
    _t = BASELINE[BASELINE["cell"] == _cell_name]
    _labels = [RANKER_LABEL[_r] for _r in BASELINE_RANKERS]
    _base = float(_t.loc[_t["ranker_key"] == MODEL_RANKER, "base_rate_permonth_mean"].iloc[0])

    print("\n" + "=" * 78)
    print(f"BASELINE COMPARISON  |  {_cell_name}")
    print("=" * 78)
    print(f"base rate {_base:.3%} per month, {len(BOARDS[_cell_name]):,} rows over "
          f"{int(BOARDS[_cell_name]['dim_month'].nunique())} month(s)")

    _lft = _t.pivot(index="k_label", columns="ranker",
                    values="lift_over_base").reindex(_k_labels)
    for _metric, _title, _with_lift in (("precision_at_k", "precision@k (lift over base)", True),
                                        ("recall_at_k", "recall@k", False)):
        _piv = _t.pivot(index="k_label", columns="ranker", values=_metric).reindex(_k_labels)
        print(f"\n  {_title}")
        print(f"  {'k':>6}" + "".join(f"{_c:>{_W}}" for _c in _labels))
        for _kl in _k_labels:
            _cells = ""
            for _c in _labels:
                _v = _piv.loc[_kl, _c]
                _cells += (f"{_v:.2%} ({_lft.loc[_kl, _c]:.1f}x)".rjust(_W)
                           if _with_lift else f"{_v:.2%}".rjust(_W))
            print(f"  {_kl:>6}{_cells}")

    print("\n  pool actually used (a ranker does not rank what it cannot score, and "
          "its own")
    print("  months carry their own base rate - which is what its lift above is "
          "against)")
    print(f"  {'ranker':<26}{'scorable':>10}{'share':>8}{'months':>12}{'base':>8}"
          f"{'k used @' + str(K_BASELINE[-1]):>13}{'short mo':>10}")
    for _r in BASELINE_RANKERS:
        _c = BASELINE_COVERAGE[(BASELINE_COVERAGE["cell"] == _cell_name)
                               & (BASELINE_COVERAGE["ranker_key"] == _r)].iloc[0]
        _last = _t[(_t["ranker_key"] == _r) & (_t["k_requested"] == K_BASELINE[-1])].iloc[0]
        _mo = f"{int(_c['months_with_any_scorable'])} of {int(_c['months_in_cell'])}"
        print(f"  {RANKER_LABEL[_r]:<26}{int(_c['scorable']):>10,}"
              f"{_c['scorable_share']:>8.1%}{_mo:>12}"
              f"{_last['base_rate_permonth_mean']:>8.2%}"
              f"{_last['k_used_mean']:>13,.0f}{int(_last['months_short_skipped']):>10}")

    _f = next(_x for _x in RHS_FLOOR if _x["cell"] == _cell_name)
    print(f"\n  tied floor of both RHS rankings ({RHS_SCORE_COL} == 0, delta == -1.0): "
          f"{_f['floor_rows_per_month']:,.0f} rows/month,")
    print(f"  precision {_f['floor_precision']:.2%}, recall {_f['floor_recall']:.2%} - below "
          "that size both RHS lists are one")
    print("  arbitrary draw from the same block, so their precision is a tie-break, "
          "not a ranking.")

# --- did the model beat both, at the list size a rep actually works? --------
print("\n" + "=" * 78)
print(f"HEADLINE  |  precision@{K_BASELINE_HEADLINE}, per month then averaged")
print("=" * 78)
for _cell_name in READ_FRAMES:
    _h = BASELINE[(BASELINE["cell"] == _cell_name)
                  & (BASELINE["k_requested"] == K_BASELINE_HEADLINE)]
    _m = float(_h.loc[_h["ranker_key"] == MODEL_RANKER, "precision_at_k"].iloc[0])
    _rhs = {_r: float(_h.loc[_h["ranker_key"] == _r, "precision_at_k"].iloc[0])
            for _r in (RHS_SCORE_RANKER, RHS_DELTA_RANKER)}
    print(f"\n{_cell_name}")
    print(f"  {'model':<26}{_m:>8.2%}")
    for _r, _v in _rhs.items():
        _ratio = f"   model is {_m / _v:.2f}x it" if _v else "   (zero - no ratio)"
        print(f"  {RANKER_LABEL[_r]:<26}{_v:>8.2%}{_ratio}")
    _beat = _m > max(_rhs.values())
    print("  -> the model BEATS both RHS rankers here." if _beat
          else "  -> the model DOES NOT beat both RHS rankers here. That is the "
               "ship gate failing, not a\n     formatting problem - read §14a's "
               "coverage table and §12's sweep before anything else.")


BASELINE COMPARISON  |  holdout_school (report half, 4717 schools)
base rate 3.269% per month, 45,109 rows over 12 month(s)

  precision@k (lift over base)
       k                   model  RHS lowest score first  RHS biggest drop first
      40           25.42% (7.8x)           11.67% (3.6x)           10.83% (5.0x)
      50           23.17% (7.1x)           11.33% (3.5x)           11.78% (5.4x)
     200           18.88% (5.8x)           13.29% (4.1x)            8.72% (4.0x)
     500           13.08% (4.0x)           11.77% (3.6x)            5.24% (2.4x)
    1000            8.83% (2.7x)            8.04% (2.5x)            3.37% (1.5x)

  recall@k
       k                   model  RHS lowest score first  RHS biggest drop first
      40                   8.25%                   3.91%                   6.17%
      50                   9.46%                   4.73%                   8.44%
     200                  31.11%                  21.83%                  25.06%
     500             

### 14c. The trigger rule as it is actually used

The team's documented lead-list rule is a **set**, not a ranking: every school whose Retention Health
Score has dropped more than 20% against its own trailing three months -
`delta_product_score_vs_t3m < RHS_TRIGGER_THRESHOLD`. The list size is whatever that comes to.

Reported per month and then averaged: how many schools it flags, its precision, its recall - and the
model's precision and recall **at that same list size**, month by month, so the two lists are the same
shape. Both are measured against the whole month, not against the rule's own scorable subset: a
school the rule cannot score is a school it never flags, and that is part of what the rule costs.

The flagged count is also divided by a rep's working capacity of 40-50 schools a month, which is the
number that says whether the rule is a lead list or a mailing list.

Writes `..._baseline_comparison.csv` - §14b's ranker rows and these trigger rows in one schema, each
carrying `k`, precision, recall, the effective list size, the base rate, the lift and the ranker's
scorable share.

In [59]:
RHS_TRIGGER_THRESHOLD = -0.20
RHS_TRIGGER_LABEL = f"trigger(delta<{RHS_TRIGGER_THRESHOLD:.2f})"
TRIGGER_POPULATION = "whole_month"


def trigger_month_table(board: pd.DataFrame, model_key: str) -> pd.DataFrame:
    """Per month: the rule as a fixed set, and the model at that same size.

    The rule's list size varies month to month, so the model's k is taken from
    it each month rather than fixed - that is what makes the two comparable.

    A row whose delta is null compares False and is therefore never flagged,
    which is the rule as it actually behaves. Precision and recall are against
    the WHOLE month either way: a school the rule cannot score is a school it
    misses, and hiding that in the denominator would flatter it.
    """
    rows = []
    for _m, _g in board.groupby("dim_month", sort=True):
        _flagged = _g.loc[_g["rhs_delta"] < RHS_TRIGGER_THRESHOLD]
        _n = len(_flagged)
        _pos = int(_g[Y_COL].sum())
        _top = (_g.sort_values(f"score__{model_key}", ascending=False, kind="mergesort")
                  .head(_n))
        _rule_hits = int(_flagged[Y_COL].sum())
        _model_hits = int(_top[Y_COL].sum())
        rows.append({
            "dim_month": _m,
            "rows": len(_g),
            "positives": _pos,
            "flagged": _n,
            "flagged_share": _n / len(_g) if len(_g) else np.nan,
            "rule_hits": _rule_hits,
            "rule_precision": _rule_hits / _n if _n else np.nan,
            "rule_recall": _rule_hits / _pos if _pos else np.nan,
            "model_hits": _model_hits,
            "model_precision": _model_hits / _n if _n else np.nan,
            "model_recall": _model_hits / _pos if _pos else np.nan,
        })
    return pd.DataFrame(rows)


def trigger_summary(tt: pd.DataFrame, cell_label: str, ranker: str, ranker_key: str,
                    prec_col: str, rec_col: str, hits_col: str) -> dict:
    """A trigger month table folded into rank_at_k's row schema."""
    _base = float((tt["positives"] / tt["rows"]).mean())
    _prec = float(tt[prec_col].mean(skipna=True))
    return {
        "cell": cell_label,
        "ranker": ranker,
        "ranker_key": ranker_key,
        "population": TRIGGER_POPULATION,
        "k_label": RHS_TRIGGER_LABEL,
        "k_requested": -1,             # not a top-k: the rule sets its own size
        "k_used_mean": float(tt["flagged"].mean()),
        "months": int(len(tt)),
        "months_short_skipped": 0,     # a fixed set, not a top-k: nothing to be short of
        "rows": int(tt["rows"].sum()),
        "positives": int(tt["positives"].sum()),
        "base_rate_permonth_mean": _base,
        "base_rate_pooled": float(tt["positives"].sum() / tt["rows"].sum()),
        "precision_at_k": _prec,
        "recall_at_k": float(tt[rec_col].mean(skipna=True)),
        "found_per_month": float(tt[hits_col].mean()),
        "lift_over_base": _prec / _base if _base else np.nan,
    }


TRIGGER, _trigger_rows = {}, []

for _cell_name in READ_FRAMES:
    _tt = trigger_month_table(BOARDS[_cell_name], MODEL_RANKER)
    TRIGGER[_cell_name] = _tt
    _trigger_rows.append(trigger_summary(
        _tt, _cell_name, RANKER_LABEL[RHS_DELTA_RANKER] + " AS A FIXED SET",
        RHS_DELTA_RANKER, "rule_precision", "rule_recall", "rule_hits"))
    _trigger_rows.append(trigger_summary(
        _tt, _cell_name, RANKER_LABEL[MODEL_RANKER] + " AT THE SAME LIST SIZE",
        MODEL_RANKER, "model_precision", "model_recall", "model_hits"))

    assert (_tt["flagged"] <= _tt["rows"]).all(), "more flagged than rows in a month"
    assert (_tt["model_hits"] <= _tt["positives"]).all(), (
        "the model found more at-risk schools in a month than the month has"
    )
    _scorable = int(BOARDS[_cell_name]["rhs_delta"].notna().sum())
    assert int(_tt["flagged"].sum()) <= _scorable, (
        f"on {_cell_name!r} the rule flagged {int(_tt['flagged'].sum()):,} rows "
        f"but only {_scorable:,} have a computable delta. A null delta must "
        "compare False, so a flag on one means the threshold is being applied to "
        "a filled value."
    )

TRIGGER_TABLE = pd.DataFrame(_trigger_rows)

# --- per month, then the two-line block ------------------------------------
print("=" * 78)
print(f"THE TRIGGER RULE AS USED  |  {RHS_DELTA_COL} < {RHS_TRIGGER_THRESHOLD:.2f}")
print("=" * 78)
for _cell_name in READ_FRAMES:
    _tt = TRIGGER[_cell_name]
    print(f"\n{_cell_name}   {_tt['rows'].mean():,.0f} rows/month, "
          f"{_tt['positives'].mean():.0f} at risk/month")
    print(f"  {'month':<10}{'rows':>9}{'flagged':>9}{'rule P':>9}{'rule R':>9}"
          f"{'model P':>10}{'model R':>9}")
    for _, _r in _tt.iterrows():
        _pc = f"{_r['rule_precision']:>9.2%}" if _r["flagged"] else f"{'n/a':>9}"
        _mc = f"{_r['model_precision']:>10.2%}" if _r["flagged"] else f"{'n/a':>10}"
        print(f"  {_r['dim_month']:%Y-%m}   {int(_r['rows']):>9,}{int(_r['flagged']):>9,}"
              f"{_pc}{_r['rule_recall']:>9.2%}{_mc}{_r['model_recall']:>9.2%}")
    _empty = int(_tt["flagged"].eq(0).sum())
    if _empty:
        print(f"  {_empty} of {len(_tt)} month(s) flagged nothing - no school has three "
              "prior months yet. Precision is")
        print("  undefined there and excluded from the mean; recall counts as 0, which "
              "is what the rule delivers.")

print("\n" + "=" * 78)
print("THE TRIGGER AGAINST THE MODEL AT THE SAME LIST SIZE")
print("=" * 78)
for _cell_name in READ_FRAMES:
    _tt = TRIGGER[_cell_name]
    _n = float(_tt["flagged"].mean())
    print(f"\n{_cell_name}   flags {_n:,.0f} schools/month "
          f"({_tt['flagged_share'].mean():.1%} of the month, "
          + ", ".join(f"{_n / _c:.1f}x a {_c}-school list" for _c in REP_CAPACITY) + ")")
    print(f"  rule    precision {_tt['rule_precision'].mean(skipna=True):>7.2%}   "
          f"recall {_tt['rule_recall'].mean(skipna=True):>7.2%}")
    print(f"  model   precision {_tt['model_precision'].mean(skipna=True):>7.2%}   "
          f"recall {_tt['model_recall'].mean(skipna=True):>7.2%}   "
          f"at the same {_n:,.0f}-school list")

# --- persist ----------------------------------------------------------------
BASELINE_OUT = pd.concat([BASELINE, TRIGGER_TABLE], ignore_index=True).merge(
    BASELINE_COVERAGE[["cell", "ranker_key", "scorable", "scorable_share",
                       "months_in_cell", "months_with_any_scorable"]],
    on=["cell", "ranker_key"], how="left")

assert BASELINE_OUT["scorable_share"].notna().all(), (
    "a comparison row has no scorable share attached, so ranker_key did not "
    "match the coverage table:\n"
    + BASELINE_OUT.loc[BASELINE_OUT["scorable_share"].isna(),
                       ["cell", "ranker", "ranker_key", "k_label"]].to_string(index=False)
)

BASELINE_CSV = RAW_DIR / f"{STEM}_baseline_comparison.csv"
BASELINE_OUT.to_csv(BASELINE_CSV, index=False)

_back = pd.read_csv(BASELINE_CSV)
assert len(_back) == len(BASELINE_OUT), "the comparison CSV does not round trip"
assert set(_back["population"]) == {BASELINE_POPULATION, TRIGGER_POPULATION}, (
    f"unexpected population labels in the CSV: {sorted(set(_back['population']))} "
    f"(expected {sorted({BASELINE_POPULATION, TRIGGER_POPULATION})})"
)
assert set(_back["k_label"]) == set(str(_k) for _k in K_BASELINE) | {RHS_TRIGGER_LABEL}, (
    f"unexpected k labels in the CSV: {sorted(set(_back['k_label']))}"
)

print(f"\nwrote  {BASELINE_CSV.name}   ({BASELINE_CSV.stat().st_size / 1e3:.1f} KB)   "
      f"{len(BASELINE_OUT)} rows")
print(f"       {len(BASELINE)} ranker rows over k = {list(K_BASELINE)} + "
      f"{len(TRIGGER_TABLE)} trigger rows, both cells")

THE TRIGGER RULE AS USED  |  delta_product_score_vs_t3m < -0.20

holdout_school (report half, 4717 schools)   3,759 rows/month, 122 at risk/month
  month          rows  flagged   rule P   rule R   model P  model R
  2024-01       3,545        0      n/a    0.00%       n/a    0.00%
  2024-02       3,587        0      n/a    0.00%       n/a    0.00%
  2024-03       3,644        0      n/a    0.00%       n/a    0.00%
  2024-04       3,663      252    9.92%   16.34%    21.83%   35.95%
  2024-05       3,669      243    9.05%   15.60%    16.87%   29.08%
  2024-06       3,710      493    4.67%   17.69%    10.95%   41.54%
  2024-07       3,742      515    4.27%   19.64%    10.29%   47.32%
  2024-08       3,818      300    7.00%   21.21%    15.33%   46.46%
  2024-09       3,870      257   10.12%   23.85%    19.07%   44.95%
  2024-10       3,918      268    8.21%   18.97%    19.03%   43.97%
  2024-11       3,949      383    4.70%   15.65%    13.84%   46.09%
  2024-12       3,994      394    3.05

---

## 15. Audit, and what is deliberately not here

Re-reads `SPLIT_ACCESS_LOG` - every `cell()` call this run made - and asserts the ship gate is absent
from it. Together with §6's static scan of the code cells and `cell()` refusing the value at runtime,
that is the three mechanisms closing on the same fact: **`test` was not loaded**.

Deliberately not in this notebook, and not accidentally missing:

- **No billing ablation.** Deferred. The billing features are in the candidate set and BorutaSHAP
  judged them like any other.
- **No hyperparameter search.** `scale_pos_weight` is printed with the note that the spec says to
  tune it downward on AUCPR; nothing here tunes it.
- **No `test`-cell number.** That is the ship gate, read once, later, on a model that is final. §14's
  baseline comparison is on `holdout_school` and `holdout_time`, so it is a read on the operating
  point, not the ship-gate number itself.

Why any of it is the way it is: `Proactive Churn/Design Decisions.md` in the Obsidian vault.

In [60]:
_accessed = sorted(set(SPLIT_ACCESS_LOG))

assert SHIP_GATE_SPLIT not in _accessed, (
    f"the ship-gate cell {SHIP_GATE_SPLIT!r} was loaded {SPLIT_ACCESS_LOG.count(SHIP_GATE_SPLIT)} "
    "time(s) in this run. cell() is supposed to refuse it, so either the "
    "gatekeeper was edited or rows were selected without it."
)
for _s in EXCLUDED_SPLITS:
    assert _s not in _accessed, f"the excluded cell {_s!r} was loaded in this run"

assert set(_accessed) <= set(LOADABLE_SPLITS), (
    f"cells loaded that are not loadable: {sorted(set(_accessed) - set(LOADABLE_SPLITS))}"
)
assert FIT_SPLIT in _accessed, (
    f"the {FIT_SPLIT!r} cell was never loaded - did §7 onwards actually run?"
)

print("split access audit")
for _s in ALL_SPLITS:
    _n = SPLIT_ACCESS_LOG.count(_s)
    _role = ("FIT + SELECT" if _s == FIT_SPLIT else
             "read" if _s in READ_SPLITS else
             "SHIP GATE" if _s == SHIP_GATE_SPLIT else "excluded")
    print(f"  {_s:<16}{_role:<14}{_n:>4} cell() call(s)")

print(f"\nOK - {SHIP_GATE_SPLIT!r} loaded 0 times, {list(EXCLUDED_SPLITS)} loaded 0 times.")
print("Three mechanisms agree: cell() refuses the value, §6's static scan found no")
print("code cell comparing against it or passing it, and this log is empty of it.")

print(f"\nartefacts in {RAW_DIR}:")
for _p in (SELECTION_JSON, IMPORTANCE_CSV, EVAL_CSV, K_SWEEP_CSV, BASELINE_CSV):
    print(f"  {_p.name:<64}{_p.stat().st_size / 1e3:>8.1f} KB")

print(f"\ncandidates {len(MODEL_COLS)} -> accepted {len(ACCEPTED)} / "
      f"tentative {len(TENTATIVE)} / rejected {len(REJECTED)} "
      f"in {BORUTA_SECONDS:.1f}s over {BORUTA_N_TRIALS} trials")

_hl = BASELINE_OUT[(BASELINE_OUT["cell"] == "holdout_time")
                   & (BASELINE_OUT["k_requested"] == K_BASELINE_HEADLINE)]
print(f"\nprecision@{K_BASELINE_HEADLINE} on holdout_time, per month then averaged:")
for _, _r in _hl.iterrows():
    print(f"  {_r['ranker']:<40}{_r['precision_at_k']:>8.2%}  "
          f"lift {_r['lift_over_base']:.2f}x  "
          f"on {_r['scorable_share']:.0%} of the cell's rows")
_tt = TRIGGER["holdout_time"]
print(f"\nthe incumbent trigger on holdout_time: {_tt['flagged'].mean():,.0f} schools/month at "
      f"{_tt['rule_precision'].mean(skipna=True):.2%} precision / "
      f"{_tt['rule_recall'].mean(skipna=True):.2%} recall;")
print(f"the model at that same list size: "
      f"{_tt['model_precision'].mean(skipna=True):.2%} / "
      f"{_tt['model_recall'].mean(skipna=True):.2%}.")
print("\nNext: settle the flagged features in §13 with a human, then the billing")
print("ablation - deferred.")

split access audit
  train           FIT + SELECT     3 cell() call(s)
  holdout_school  read             1 cell() call(s)
  holdout_time    read             1 cell() call(s)
  test            SHIP GATE        0 cell() call(s)
  embargo         excluded         0 cell() call(s)
  retrain_only    excluded         0 cell() call(s)

OK - 'test' loaded 0 times, ['embargo', 'retrain_only'] loaded 0 times.
Three mechanisms agree: cell() refuses the value, §6's static scan found no
code cell comparing against it or passing it, and this log is empty of it.

artefacts in /Users/swami/Desktop/Projects/Proactive_Churn/data/raw:
  churn_v1_is_reportable_request_202401_202605_h3_boruta_selection.json     9.7 KB
  churn_v1_is_reportable_request_202401_202605_h3_boruta_importance.csv    16.5 KB
  churn_v1_is_reportable_request_202401_202605_h3_selection_eval.csv     2.8 KB
  churn_v1_is_reportable_request_202401_202605_h3_k_sweep.csv          8.4 KB
  churn_v1_is_reportable_request_202401_202605_h3_b